# Reproducing Kussovska, Kim and Rungratsameetaweemana (2025) from dandiset 000469

Paper. Kussovska D, Kim R, Rungratsameetaweemana N (2025). Neural heterogeneity shapes the
temporal structure of human working memory. bioRxiv,
[doi:10.1101/2025.10.31.684900](https://doi.org/10.1101/2025.10.31.684900). The claims file
(`claims/kussovska-2025.yaml`) was written from the bioRxiv full text posted on 1 November 2025
(main text, figure legends and Methods; the Extended Data figures and their legends were not in
the text read).

Dandiset. [DANDI:000469](https://dandiarchive.org/dandiset/000469/0.231213.2047), version
0.231213.2047 (Kyzar et al. 2024, human single neurons during a Sternberg working memory task).
This is the version the paper cites (Methods, Participants and data source, and Data
availability). The four published versions (0.231213.2047, 0.240122.1800, 0.240122.1812,
0.240123.1806) hold the same 41 assets with the same asset identifiers and sizes, which the data
check below confirms, so the version the paper's code downloads (0.240123.1806) gives the same
data.

Paper's code. github.com/dariakussovska/heterogeneitywm at commit 5144fe8, the commit the
claims file read. Four of its shipped tables are outside inputs here:
`data/new_trial_info.xlsx` (the 96 load-matched trials of the GLM),
`data/new_trial_final.xlsx` (the 75 identity-matched trials of the pseudo-population decoding),
both of which the claims file names as readings, and, for comparison only (no outcome rests on
them), `data/all_neuron_brain_regions_cleaned.xlsx` (the code's concept-cell list) and
`data/cell_analysis/Cell_metrics.xlsx` (the code's ACG features). The rodent validation claim uses
`data/cell_analysis/validation_data.xlsx`, the features of the CellExplorer ground-truth cells
(`data: elsewhere`).

| Claim | Tested here |
|---|---|
| units-by-region, behavior-accuracy, fig1c-accuracy-load1-vs-load3, behavior-rt, rt-in-vs-out-memory, rt-correct-vs-incorrect | tested |
| fig1e-load-modulated-neurons, fig1f-rt-modulation-not-significant | tested |
| concept-cells-count, fig2ab-single-cells-not-persistent, fig2c-population-persistence, ext1bc-persistence-by-load | tested |
| fig2d-cross-temporal-localized, ext1de-cross-temporal-load23 | tested |
| fig2e-maintenance-decoding-amy-mtl, ext1f-maintenance-decoding-load2 | tested |
| fig2f-bin-width-decoding, fig2f-bin-width-pairwise, ext1g-bin-width-load2 | tested |
| fig3abc-trajectory-complexity, fig3d-encoding-vs-maintenance-separability, fig3h-early-vs-late-maintenance | blocked (too large; the time of one dPCA fit is measured below) |
| fig4c-encoding-bursts-vs-poisson, fig4d-maintenance-bursts-vs-poisson, fig4-acg-group-differences, fig4f-all-cells-epochs, fig4hj-group-epochs, ext2-bursts-loads23 | tested |
| acg-fit-yield, fig5c-cell-type-clusters, fig5dfh-cluster-features, fig5gh-burst-index-cv2, fig5kl-concept-cell-types | tested |
| supp4-rodent-validation | tested on the features shipped with the paper's code (`data: elsewhere`) |
| fig6ab-celltype-bursts-vs-null, fig6cd-celltype-epochs, ext5bd-in-load-dependence | tested |
| fig6e-load-modulation-in, fig6f-rt-modulation-in-concept, fig6gh-in-concept-cv2, ext5gh-py-decay-preferred | tested |
| fig6i-py-bin-width-decoding, fig6j-in-bin-width-decoding, fig6kl-cross-temporal-by-type, ext5ij-load2-decoding-by-type | tested |

Each claim's section quotes it, runs every reading its criterion names, and ends with a
conclusion. Random generators are seeded per section.

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from scipy import stats
from scipy.stats import wilcoxon, mannwhitneyu, kruskal, binom, binomtest
from scipy import ndimage

import wm

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
T0 = time.time()

sessions = wm.load_all()
BY = {s["sub"]: s for s in sessions}
NT = wm.neuron_table(sessions)
T = {s["sub"]: wm.trial_frame(s) for s in sessions}
SPK = [wm.spikes_of(BY, r) for _, r in NT.iterrows()]
FIX = [wm.fixation_stats(SPK[n], T[NT["sub"].iat[n]]) for n in range(len(NT))]
print(f"streamed {len(sessions)} Sternberg files of dandiset {wm.DANDISET_ID} version {wm.DANDISET_VERSION} "
      f"in {time.time() - T0:.0f} s")

# Outside inputs from the paper's code repository at the pinned commit.
INFO = wm.code_table("data/new_trial_info.xlsx")
FINAL = wm.code_table("data/new_trial_final.xlsx")
SHIP_REG = wm.code_table("data/all_neuron_brain_regions_cleaned.xlsx")
SHIP_CM = wm.code_table("data/cell_analysis/Cell_metrics.xlsx")
VALID = wm.code_table("data/cell_analysis/validation_data.xlsx")
print("code tables:", {k: v.shape for k, v in dict(INFO=INFO, FINAL=FINAL, SHIP_REG=SHIP_REG,
                                                  SHIP_CM=SHIP_CM, VALID=VALID).items()})

## Data check

Each check of the description against the values is stated under its output.

In [ ]:
# Versions: the same assets in every published version.
vers = [v["version"] for v in requests.get(f"{wm.API}/dandisets/{wm.DANDISET_ID}/versions/").json()["results"]]
listing = {v: wm.list_assets(v) for v in vers}
sig = {v: sorted((a["path"], a["asset_id"], a["size"]) for a in listing[v]) for v in vers}
print("versions:", vers)
for v in vers:
    print(f"  {v}: {len(listing[v])} assets, {sum(a['size'] for a in listing[v]) / 1e9:.2f} GB, "
          f"same assets as {wm.DANDISET_VERSION}: {sig[v] == sig[wm.DANDISET_VERSION]}")
paths = [a["path"] for a in listing[wm.DANDISET_VERSION]]
print("files per session type:", pd.Series([p.split("_")[1] for p in paths]).value_counts().to_dict(),
      "; subjects:", len({p.split("/")[0] for p in paths}))
print("paper: 21 patients, 41 recording sessions")

The pinned version holds 41 files, a screening session (`ses-1`) and a Sternberg session
(`ses-2`) for each of 21 subjects except sub-19, which has only `ses-2`: 41 sessions from 21
patients, as the paper says. Every published version holds the same assets. The analysis reads
only the 21 Sternberg files, as the paper's code does.

In [ ]:
subj = pd.DataFrame([{"sub": s["sub"], "age": s["subject"]["age"], "sex": s["subject"]["sex"],
                      "units": len(s["units"]), "trials": len(s["trials"]),
                      "description": s["session_description"][:30]} for s in sessions])
ages = subj.age.str.extract(r"P(\d+)Y")[0].astype(int)
print(subj.to_string(index=False))
print(f"\nage {ages.min()} to {ages.max()} years, mean {ages.mean():.2f} +/- {ages.std():.2f} (s.d.); "
      "paper: 17 to 71, 36.10 +/- 15.45 (over 41 sessions)")
print("trials per session:", subj.trials.value_counts().to_dict(), "; paper: 16 participants with 135 trials, 5 with 108")
print("units:", len(NT), "; by region:", NT.region.value_counts().to_dict())
print("location labels:", sorted(NT.location.unique()))

The files hold 902 units (as the paper says) in five regions, each with a left and a right label.
Fifteen sessions have 135 trials and six have 108 (subjects 6, 7, 8, 9, 16 and 19); the paper
says sixteen and five, so the data disagree with the description there. The ages run from 17 to
70 (mean 36.19); the paper gives 17 to 71 and 36.10 over sessions, which one Sternberg file per
subject cannot check.

In [ ]:
tr = pd.concat(T.values())
rows = []
for sub, t in T.items():
    rows.append({"sub": sub, "n": len(t), "load1": (t.load == 1).sum(), "load2": (t.load == 2).sum(),
                 "load3": (t.load == 3).sum(), "images": sorted(set(t.img1)),
                 "fix_s": f"{(t.enc1 - t.fix_start).min():.3f}-{(t.enc1 - t.fix_start).max():.3f}",
                 "enc1_s": f"{(t.timestamps_Encoding1_end - t.enc1).min():.3f}-{(t.timestamps_Encoding1_end - t.enc1).max():.3f}",
                 "maint_s": f"{t.maint_dur.min():.3f}-{t.maint_dur.max():.3f}",
                 "rt_s": f"{t.rt.min():.2f}-{t.rt.max():.2f}",
                 "ordered": bool(((t.enc1 > t.fix_start) & (t.maint > t.enc1) & (t.probe > t.maint)
                                  & (t.timestamps_Response > t.probe)).all())})
print(pd.DataFrame(rows).to_string(index=False))
print("\nimage ids at encoding 2 and 3 on load 1 trials (unused positions):",
      sorted(set(tr[tr.load == 1].img2)), sorted(set(tr[tr.load == 1].img3)))
print("accuracy values:", sorted(tr.correct.unique()), "; probe_in_out values:", sorted(tr.in_memory.unique()))
print(f"maintenance duration over all trials {tr.maint_dur.min():.3f} to {tr.maint_dur.max():.3f} s "
      "(paper: 2.5 to 2.8 s); fixation "
      f"{(tr.enc1 - tr.fix_start).min():.3f} to {(tr.enc1 - tr.fix_start).max():.3f} s (paper: 0.9 to 1.0 s)")
# the second time unit check: spike times lie within the session's trial span
span = pd.DataFrame([{"sub": NT["sub"].iat[n], "first": SPK[n][0] if len(SPK[n]) else np.nan,
                      "last": SPK[n][-1] if len(SPK[n]) else np.nan} for n in range(len(NT))])
lim = pd.DataFrame([{"sub": s, "t0": t.start_time.min(), "t1": t.stop_time.max()} for s, t in T.items()])
m = span.groupby("sub").agg(first=("first", "min"), last=("last", "max")).join(lim.set_index("sub"))
print("spike times start before the first trial and end after the last in every session:",
      bool(((m["first"] <= m.t0 + 60) & (m["last"] >= m.t1 - 60)).all()),
      f"; session spans {((m['last'] - m['first']) / 60).min():.0f} to {((m['last'] - m['first']) / 60).max():.0f} min")
print("units with fewer than 2 spikes:", int((NT.n_spikes < 2).sum()))

Every session has equal numbers of trials per load and five images (ids 1 to 5) in its encoding
periods. The epoch timestamps are ordered in every trial except one: in subject 9 (the session
marked False above) one trial has a maintenance timestamp equal to its probe timestamp, a
maintenance of 0 s, and its fixation durations include 0 s. Apart from that trial, maintenance
lasts about 2.53 to 2.84 s (paper: 2.5 to 2.8 s). Fixation lasts about 0.91 to 1.2 s in most
sessions, longer than the paper's 0.9 to 1.0 s, with single long fixations (14.07 s in subject 21).
The first encoding image lasts 1.0, 1.2, 1.5 or 2.5 s depending on the session (enc1_s); the
paper does not state the image duration, and the code's 1 s encoding window is used throughout.
Spike times are in seconds and span each session's trials; the contrast below checks the clock.

In [ ]:
# Clock check by contrast: image selectivity of each neuron 200 to 1000 ms after the first
# encoding onset (the concept-cell window) against the same test in windows shifted 1 s and 2 s
# earlier (fixation and the previous trial's probe), per session. With aligned clocks only the
# true window should hold image information beyond the false-positive rate.
rng = np.random.default_rng(1)
rows = []
for sub, t in T.items():
    nids = NT.nid[NT["sub"] == sub].values
    row = {"sub": sub, "neurons": len(nids)}
    for shift in [0.0, -1.0, -2.0]:
        k = 0
        for n in nids:
            r = wm.count_in(SPK[n], (t.enc1 + 0.2 + shift).values, (t.enc1 + 1.0 + shift).values) / 0.8
            k += wm.concept_test(r, t.img1.values, rng, reading="code")[2]
        row[f"selective_at_{shift:+.0f}s"] = k
    # evoked response: population rate 0-300 ms after onset over the 300 ms before
    pre = sum(wm.count_in(SPK[n], (t.enc1 - 0.3).values, t.enc1.values).sum() for n in nids)
    post = sum(wm.count_in(SPK[n], (t.enc1 + 0.2).values, (t.enc1 + 0.5).values).sum() for n in nids)
    row["rate_200_500ms_over_pre"] = round(post / max(pre, 1), 2)
    rows.append(row)
clk = pd.DataFrame(rows)
print(clk.to_string(index=False))
print("totals:", clk[[c for c in clk.columns if c.startswith("selective")]].sum().to_dict())

Over all sessions, 58 neurons are image-selective in the window after the first encoding onset,
against 11 and 16 in the windows shifted 1 s and 2 s earlier, so image information appears where
the trials table puts the images. This is consistent with spike times and trials sharing a clock;
it is shown per session only where selective neurons exist. Subjects 1, 3, 6, 10, 17, 19 and 20
have no selective neuron in the true window, and the population rate barely changes after image
onset in any session (ratios 0.89 to 1.14), so for those seven sessions the alignment could not
be established.

In [ ]:
# Duplicate recordings: identical spike trains or identical trial tables across sessions.
keys = {}
dups = []
for n in range(len(NT)):
    s = SPK[n]
    k = (len(s), round(float(s[:5].sum()), 6) if len(s) else 0.0)
    if k in keys and NT["sub"].iat[keys[k]] != NT["sub"].iat[n]:
        dups.append((keys[k], n))
    keys.setdefault(k, n)
print("units with an identical spike train in another session:", len(dups))
tabs = {}
for sub, t in T.items():
    key = tuple(np.round(t.enc1.values[:10] - t.enc1.values[0], 4))
    tabs.setdefault(key, []).append(sub)
print("sessions with identical trial timing:", [v for v in tabs.values() if len(v) > 1])
seqs = {}
for sub, t in T.items():
    seqs.setdefault(tuple(t.img1.values) + tuple(t.load.values), []).append(sub)
print("sessions sharing the same trial sequence (loads and image ids):", [v for v in seqs.values() if len(v) > 1])

No unit or trial timing is duplicated across sessions. Several sessions share one trial sequence
(the same loads and image ids in the same order), which is expected if the task used a fixed
pseudorandom list; their timings differ, so these are separate recordings.

In [ ]:
# Link of the code's trial tables to the dataset's trials (needed by the readings that use them):
# trial_id is the row number, and their loads, image indices and reaction times agree.
rows = []
for sub, t in T.items():
    ti = t.set_index("trial")
    for name, tab, cols in [("new_trial_info", INFO, ("num_images_presented", "RT")),
                            ("new_trial_final", FINAL, ("num_images_presented", "RT "))]:
        x = tab[tab.subject_id == sub]
        m = ti.loc[x.trial_id.values]
        row = {"table": name, "sub": sub, "rows": len(x), "loads_agree": bool((x[cols[0]].values == m.load.values).all()),
               "rt_max_diff": float(np.abs(x[cols[1]].values - m.rt.values).max())}
        if name == "new_trial_final":
            mp = pd.DataFrame({"code": x.stimulus_index_enc1.values, "pic": m.img1.values})
            row["image_map_one_to_one"] = bool(mp.groupby("code").pic.nunique().max() == 1
                                               and mp.groupby("pic").code.nunique().max() == 1)
        rows.append(row)
lk = pd.DataFrame(rows)
print(lk.groupby("table").agg(sessions=("sub", "size"), rows_min=("rows", "min"), rows_max=("rows", "max"),
                              loads_agree=("loads_agree", "all"), rt_max_diff=("rt_max_diff", "max")).to_string())
print("image index of new_trial_final maps one to one onto the dataset's picture ids in every session:",
      bool(lk.image_map_one_to_one.dropna().all()))
f1 = FINAL[FINAL.num_images_presented == 1]
print("new_trial_final load-1 rows per session:", f1.groupby("subject_id").size().unique(),
      "; image index the same across sessions in every load-1 row:",
      bool((f1.groupby("trial_id_final").stimulus_index_enc1.nunique() == 1).all()))
for L in [2, 3]:
    fl = FINAL[FINAL.num_images_presented == L]
    print(f"load {L} rows per session: {fl.groupby('subject_id').size().unique()}; rows whose label image "
          f"(position {L}) has the same index in every session: "
          f"{(fl.groupby('trial_id_final')[f'stimulus_index_enc{L}'].nunique() == 1).mean():.2f}; "
          f"label counts in subject 14: {fl[fl.subject_id == 14][f'stimulus_index_enc{L}'].value_counts().sort_index().to_dict()}")
ids1 = INFO[INFO.subject_id == 1].new_trial_id.values
print("new_trial_info: rows per session", INFO.groupby("subject_id").size().unique(),
      "; subject 1's 96 ids present in every session:",
      all(set(ids1) <= set(INFO[INFO.subject_id == s].new_trial_id) for s in T),
      "; load the same across sessions for each id:",
      bool((INFO.groupby("new_trial_id").num_images_presented.nunique() == 1).all()))
ok = (SHIP_REG.subject_id.values == NT["sub"].values).all() and (SHIP_REG.Neuron_ID.values == NT.unit.values).all() \
     and (SHIP_REG.Location.values == NT.location.values).all()
print("the code's neuron table lists the dataset's 902 units in the same order, with the same locations:", bool(ok))

The code's tables match the dataset trial by trial: `trial_id` is the row of the trials table,
and loads and reaction times agree exactly. In the identity-matched table the image index of
every load-1 row is the same in all 21 sessions and maps one to one onto each session's picture
ids, so its rows can align a pseudo-population; its load-2 and load-3 rows are aligned on the
label image (second and third) in every session too. The code's neuron table lists the dataset's
units in order, with the same locations. Several sessions share one trial sequence, and the
subjects with 108 trials have 106 rows in new_trial_info.

## units-by-region

> Recordings from the 21 patients yielded 902 well-isolated units, 259 in the amygdala, 171 in the
> dACC, 190 in the hippocampus, 250 in the preSMA and 32 in the vmPFC.

In [ ]:
cnt = NT.region.value_counts()
print("patients:", NT["sub"].nunique(), "; units:", len(NT))
print(pd.DataFrame({"dataset": cnt, "paper": pd.Series({"AMY": 259, "dACC": 171, "HPC": 190, "preSMA": 250, "vmPFC": 32})}).to_string())
print("\nunits by participant and region:")
print(pd.crosstab(NT["sub"], NT.region).to_string())

**Result: reproduced.** 21 patients and 902 units, with exactly the paper's count in each of the
five regions (left and right merged).

## behavior-accuracy

> Task accuracy was 88.73% ± 1.25%.

In [ ]:
acc = tr.groupby("sub").correct.mean() * 100
print(acc.round(2).to_dict())
print(f"mean over the 21 participants {acc.mean():.2f}%, s.e.m. {acc.std(ddof=1) / np.sqrt(len(acc)):.2f}, "
      f"s.d. {acc.std(ddof=1):.2f}; range {acc.min():.2f} to {acc.max():.2f}")
print(f"variant, pooled over all {len(tr)} trials: {100 * tr.correct.mean():.2f}%")
print("paper: 88.73 +/- 1.25 (tolerance 87.48 to 89.98)")

**Result: reproduced.** The mean of the 21 participants' accuracies is 88.73% with s.e.m. 1.25,
the paper's value and spread (the s.d. is 5.74); the pooled variant gives 88.70%.

## fig1c-accuracy-load1-vs-load3

> Accuracy was higher for load 1 than for load 3 trials across patients.

In [ ]:
al = tr.groupby(["sub", "load"]).correct.mean().unstack()
print((100 * al).round(1).to_string())
print("median accuracy by load:", (100 * al.median()).round(2).to_dict())
H, pk = kruskal(al[1], al[2], al[3])
print(f"Kruskal-Wallis over loads H = {H:.3f}, p = {pk:.4g}")
for a, b in [(1, 2), (1, 3), (2, 3)]:
    u = mannwhitneyu(al[a], al[b], alternative="two-sided")
    print(f"load {a} vs {b}: rank-sum U = {u.statistic:.1f}, p raw = {u.pvalue:.4f}, Bonferroni x3 = {min(1, 3 * u.pvalue):.4f}")
w = wilcoxon(al[1], al[3])
print(f"variant, paired signed-rank load 1 vs 3: W = {w.statistic:.1f}, p = {w.pvalue:.4f}; "
      f"load 1 higher in {(al[1] > al[3]).sum()} of 21, equal in {(al[1] == al[3]).sum()}")
print("paper: rank-sum P = 0.027")

**Result: reproduced.** Load 1 accuracy (median 94.44%) is higher than load 3 (86.67%); the
rank-sum p is 0.0090 raw and 0.0270 after the code's Bonferroni factor of 3, the paper's P = 0.027.
The paired variant gives p = 0.0042.

## behavior-rt

> Reaction time for correct trials was 1.06 ± 0.28 s.

In [ ]:
c = tr[tr.correct == 1]
med = c.groupby("sub").rt.median()
mean = c.groupby("sub").rt.mean()
q = c.rt.quantile([0.25, 0.75])
print(f"(a) mean of participants' median RT {med.mean():.3f} s (s.d. {med.std(ddof=1):.3f}); "
      f"(b) mean of participants' mean RT {mean.mean():.3f} s (s.d. {mean.std(ddof=1):.3f}); "
      f"(c) median over {len(c)} correct trials {c.rt.median():.3f} s (IQR {q.iloc[0]:.3f} to {q.iloc[1]:.3f}, half-IQR {(q.iloc[1] - q.iloc[0]) / 2:.3f}; trial s.d. {c.rt.std():.3f})")
print(f"RT range over correct trials {c.rt.min():.3f} to {c.rt.max():.3f} s")
print("paper: 1.06 +/- 0.28 s (tolerance 0.78 to 1.34)")

**Result: reproduced.** Readings (a), 1.120 s, and (c), 1.097 s, fall within 0.78 to 1.34 s;
reading (b), 1.348 s, is just above it. The spreads are 0.252 s (s.d. of participant medians)
and 0.297 s (half the IQR of trials), near the paper's 0.28.

## rt-in-vs-out-memory

> Reaction time was faster for probes held in memory than for probes not in memory.

In [ ]:
rng = np.random.default_rng(11)


def paired_signflip(d, rng, n=1000):
    d = np.asarray(d)
    null = np.abs((rng.choice([-1, 1], (n, len(d))) * d).mean(1))
    return float((np.sum(null >= abs(d.mean())) + 1) / (n + 1))


def label_perm(x, lab, rng, n=1000):
    obs = x[lab].mean() - x[~lab].mean()
    null = np.array([(lambda p: x[p].mean() - x[~p].mean())(rng.permutation(lab)) for _ in range(n)])
    return obs, float((np.sum(np.abs(null) >= abs(obs)) + 1) / (n + 1))


for name, sel in [("correct trials", tr.correct == 1), ("all trials", tr.correct >= 0)]:
    x = tr[sel]
    d = x.groupby(["sub", "in_memory"]).rt.median().unstack()
    diff = d[1] - d[0]
    pa = paired_signflip(diff.values, rng)
    obs, pb = label_perm(x.rt.values, x.in_memory.values == 1, rng)
    print(f"{name}: (a) median RT in memory {d[1].median():.3f} s, not in memory {d[0].median():.3f} s; "
          f"per-participant difference mean {diff.mean():.3f} s, faster in {(diff < 0).sum()} of {len(diff)}; sign-flip p = {pa:.4f}")
    print(f"   (b) pooled mean difference in - out = {obs:.3f} s, label permutation p = {pb:.4f}")
print("paper: permutation test P = 0.001")

**Result: reproduced.** Probes held in memory are answered faster (correct trials, medians 1.041
against 1.184 s, faster in 17 of 21 participants), with p at most 0.002 under every reading.

## rt-correct-vs-incorrect

> Correct decisions were faster than incorrect ones.

In [ ]:
rng = np.random.default_rng(12)
d = tr.groupby(["sub", "correct"]).rt.median().unstack().dropna()
diff = d[1] - d[0]
pa = paired_signflip(diff.values, rng)
obs, pb = label_perm(tr.rt.values, tr.correct.values == 1, rng)
print(f"(a) {len(d)} participants with incorrect trials; median of correct {d[1].median():.3f} s, incorrect {d[0].median():.3f} s; "
      f"correct faster in {(diff < 0).sum()} of {len(diff)}; sign-flip p = {pa:.4f}")
print(f"(b) pooled mean difference correct - incorrect = {obs:.3f} s, label permutation p = {pb:.4f}")
print("incorrect trials per participant:", tr[tr.correct == 0].groupby("sub").size().to_dict())
print("paper: permutation test P = 0.002")

**Result: reproduced.** Correct responses are faster than incorrect ones (medians 1.084 against
1.503 s, in 20 of 21 participants), p = 0.001 under both readings.

## fig1e-load-modulated-neurons

> During maintenance a significant proportion of neurons in the preSMA (19.6%), vmPFC (18.75%) and
> dACC (10.53%) was modulated by memory load.

Spike counts in the first 2.5 s of maintenance, a Poisson GLM on the z-scored predictor fitted by
Newton's method (the same maximum-likelihood fit as the code's unpenalised `PoissonRegressor`;
checked against sklearn below), 1,000 permutations with seed 17. Trial readings: (a) the code's
96 load-matched trials (`new_trial_info.xlsx`: subject 1's `new_trial_id`s, each neuron's count
from its own session's trial with the same id), with the permutation shared over the trial vector
(the code) or drawn within each participant (Methods); (b) all trials of each participant,
permuted within participant.

In [ ]:
REG = {g: NT.nid[NT.region == g].values for g in ["AMY", "HPC", "dACC", "preSMA", "vmPFC"]}
NP = 1000


def glm_design(pred, trials, perm, seed=17):
    rng = np.random.default_rng(seed)
    shared = np.array([rng.permutation(len(ids1)) for _ in range(NP)])
    subperm, Y, X, P = {}, [], [], []
    for n in range(len(NT)):
        sub = NT["sub"].iat[n]
        t = T[sub].set_index("trial")
        if trials == "matched":
            rows = INFO[INFO.subject_id == sub].set_index("new_trial_id").loc[ids1, "trial_id"].values
        else:
            rows = t.index.values
        tt = t.loc[rows]
        Y.append(wm.count_in(SPK[n], tt.maint.values, tt.maint.values + 2.5))
        if pred == "rt_code":
            X.append(INFO[INFO.subject_id == 1].set_index("new_trial_id").loc[ids1, "RT"].values)
        else:
            X.append(tt["load" if pred == "load" else "rt"].values)
        if perm == "shared":
            P.append(shared)
        else:
            if sub not in subperm:
                subperm[sub] = np.array([rng.permutation(len(rows)) for _ in range(NP)])
            P.append(subperm[sub])
    return Y, X, P


GLM = {}
for pred in ["load", "rt", "rt_code"]:
    for trials, perm in [("matched", "shared"), ("matched", "within"), ("all", "within")]:
        if pred == "rt_code" and trials == "all":
            continue
        Y, X, P = glm_design(pred, trials, perm)
        GLM[(pred, trials, perm)] = (wm.glm_fit(Y, X, P), Y, X)

# check of the fit against sklearn on 20 neurons
from sklearn.linear_model import PoissonRegressor
(real, null), Y, X = GLM[("load", "matched", "shared")]
dev = []
for n in range(0, 902, 45):
    x = (X[n] - X[n].mean()) / X[n].std()
    if Y[n].sum() > 0:
        dev.append(abs(PoissonRegressor(alpha=0, max_iter=1000).fit(x[:, None], Y[n]).coef_[0] - real[n]))
print(f"largest |slope - sklearn slope| over {len(dev)} neurons: {max(dev):.2e}")
print("neurons with no maintenance spikes in the matched trials:", int(sum(y.sum() == 0 for y in Y)))
for key in [("load", "matched", "shared"), ("load", "matched", "within"), ("load", "all", "within")]:
    (real, null), _, _ = GLM[key]
    df, _ = wm.glm_groups(real, null, REG)
    print(f"\nload, trials {key[1]}, permutation {key[2]}:")
    print(df.round(3).to_string(index=False))
print("\npaper: preSMA 19.6% (P = 0.001), vmPFC 18.75% (P = 0.009), dACC 10.53% (P = 0.026); tolerance 10% of each")

**Result: reproduced.** Under reading (a) with the code's shared permutation, preSMA (20.40%,
p = 0.000), vmPFC (18.75%, p = 0.009) and dACC (10.53%, p = 0.025) are each within tolerance with
p < 0.05; the within-participant permutation gives 19.20%, 18.75% and 9.94%, also within
tolerance. With all trials (reading b) the three regions stay significant with higher
percentages (22.80%, 21.88%, 12.87%). The amygdala and hippocampus are not significant under any
reading.

## fig1f-rt-modulation-not-significant

> No region had a significant proportion of neurons whose maintenance firing covaried with reaction
> time after correction, although the amygdala (5.79%), preSMA (8.00%) and vmPFC (6.25%) showed
> elevated percentages.

The GLM of the previous section with RT as the predictor; Benjamini-Hochberg over the five
regions. The main analysis uses each trial's own RT; the code's predictor, subject 1's RT vector
for every neuron (main_1e,f.py:113), is the variant.

In [ ]:
for key in [("rt", "matched", "shared"), ("rt", "matched", "within"), ("rt", "all", "within"),
            ("rt_code", "matched", "shared"), ("rt_code", "matched", "within")]:
    (real, null), _, _ = GLM[key]
    df, _ = wm.glm_groups(real, null, REG)
    df["p_BH"] = wm.bh(df.p)
    label = "each trial's own RT" if key[0] == "rt" else "variant, subject 1's RT vector (the code)"
    print(f"\n{label}; trials {key[1]}, permutation {key[2]}:")
    print(df.round(3).to_string(index=False))
print("\npaper: AMY 5.79%, preSMA 8.00%, vmPFC 6.25%, none significant after correction")

**Result: not reproduced.** With each trial's own reaction time, the claim's test finds regions
with a significant proportion of RT-modulated neurons after BH correction (preSMA, the amygdala
and dACC under reading (a); see the p_BH column), and the percentages are higher than the
paper's. The code's predictor, subject 1's RT vector applied to every neuron, gives the paper's
three percentages exactly (AMY 5.79%, preSMA 8.00%, vmPFC 6.25%) and no region significant after
correction under the shared permutation, so the paper's numbers come from that predictor, which
pairs each neuron's counts with another participant's reaction times.

## concept-cells-count

> Of 902 neurons, 58 (6.4%) were concept cells, 4 ± 4 across participants, 70.69% of them in the
> amygdala, 18.96% in the hippocampus, 1.72% in the dACC, 3.45% in the preSMA and 5.15% in the vmPFC.

Spike counts 200 to 1000 ms after first-encoding onset on all trials; the two images with the
highest mean rate; bootstrap of the difference (1,000 resamples). Reading (a) is the Methods'
bootstrap, (b) the code's; ten seeds each (the code sets none).

In [ ]:
CCR = {}
rates = []
for n in range(len(NT)):
    t = T[NT["sub"].iat[n]]
    rates.append((wm.count_in(SPK[n], (t.enc1 + 0.2).values, (t.enc1 + 1.0).values) / 0.8, t.img1.values))
rows = []
for reading in ["methods", "code"]:
    for seed in range(10):
        rng = np.random.default_rng(seed)
        res = pd.DataFrame([wm.concept_test(r, im, rng, reading=reading) for r, im in rates], columns=["pref", "p", "sig"])
        CCR[(reading, seed)] = res
        reg = NT.region[res.sig.values].value_counts()
        rows.append({"reading": reading, "seed": seed, "concept": int(res.sig.sum()),
                     **{f"%{g}": round(100 * reg.get(g, 0) / max(res.sig.sum(), 1), 2) for g in ["AMY", "HPC", "dACC", "preSMA", "vmPFC"]}})
cc_tab = pd.DataFrame(rows)
print(cc_tab.to_string(index=False))
print("\nrange of counts:", cc_tab.groupby("reading").concept.agg(["min", "max"]).to_dict("index"))
CONCEPT = CCR[("code", 0)].sig.values
PREF = CCR[("code", 0)].pref.values
per = pd.Series(NT["sub"].values[CONCEPT]).value_counts()
print(f"\nset used downstream: reading (b), seed 0, {CONCEPT.sum()} cells; per participant with any: "
      f"{per.mean():.2f} +/- {per.std():.2f} over {len(per)} participants; over all 21: {per.reindex(range(1, 22), fill_value=0).mean():.2f} +/- {per.reindex(range(1, 22), fill_value=0).std():.2f}")
ship = (SHIP_REG.Signi == "Y").values
print(f"for comparison, the code's shipped list: {ship.sum()} cells, {int((ship & CONCEPT).sum())} shared with the set used here; "
      f"cells in the shipped list found in k of the 10 code-reading seeds: "
      f"{pd.Series(np.array([CCR[('code', s)].sig.values for s in range(10)]).sum(0)[ship]).value_counts().sort_index().to_dict()}")
print("paper: 58 (tolerance 52.2 to 63.8); AMY 70.69% (63.6 to 77.8), HPC 18.96% (17.06 to 20.86)")

**Result: reproduced.** Under the code's bootstrap (reading b) the count is 57 to 60 over ten
seeds, all within tolerance, with the amygdala share 70.18 to 72.41% and the hippocampus share
17.24 to 19.30%; seed 0 gives the paper's regional shares exactly (41, 11, 1, 2 and 3 cells). The
Methods' bootstrap (reading a) finds 49 to 54 cells, within tolerance only for two seeds. The
per-participant spread is 4.14 +/- 3.66 over the 14 participants with a concept cell, close to
the paper's 4 +/- 4. The set used in later sections is reading (b), seed 0 (a standard choice of
seed); it shares 57 of its 58 cells with the code's shipped list.

## fig2ab-single-cells-not-persistent

> Examined neuron by neuron, most concept cells did not maintain elevated firing for their
> preferred image throughout the maintenance epoch.

Per concept cell, load 1 trials, 50 ms bins of the first 2.5 s of maintenance, z-scored to
fixation and smoothed (sigma one bin); preferred trials are those whose first image is the
cell's preferred image. Reading (a): a permutation test per bin (1,000 shuffles of the
preferred/nonpreferred labels), persistent when significantly higher on preferred trials in at
least half the bins; reading (b): persistent when the spike count is significantly higher on
preferred trials in each of the 1 s windows M1 to M3 (M3 ends at the end of maintenance).

In [ ]:
CELLS_C = np.where(CONCEPT)[0]


def pref_mask(t, pref, load):
    if load == 1:
        return (t.img1 == pref).values
    return ((t.img1 == pref) | (t.img2 == pref) | (t.img3 == pref)).values


rows = []
for n in CELLS_C:
    rng = np.random.default_rng(int(n))
    t = T[NT["sub"].iat[n]]
    t = t[t.load == 1]
    pm = pref_mask(t, PREF[n], 1)
    Z = wm.psth_z(SPK[n], t.maint.values, 2.5, *FIX[n])
    obs = Z[pm].mean(0) - Z[~pm].mean(0)
    perms = [rng.permutation(pm) for _ in range(1000)]
    null = np.array([Z[q].mean(0) - Z[~q].mean(0) for q in perms])
    p = np.mean(np.abs(null) >= np.abs(obs), 0)
    frac = np.mean((p < 0.05) & (obs > 0))
    win = []
    for i in range(3):
        c = wm.count_in(SPK[n], t.maint.values + i, np.minimum(t.maint.values + i + 1, t.probe.values))
        o = c[pm].mean() - c[~pm].mean()
        nn = np.array([c[q].mean() - c[~q].mean() for q in perms])
        win.append(bool(o > 0 and np.mean(np.abs(nn) >= abs(o)) < 0.05))
    rows.append({"nid": n, "sub": NT["sub"].iat[n], "region": NT.region.iat[n], "pref_trials": int(pm.sum()),
                 "frac_bins_higher": frac, "M1": win[0], "M2": win[1], "M3": win[2]})
sc = pd.DataFrame(rows)
sc["persistent_a"] = sc.frac_bins_higher >= 0.5
sc["persistent_b"] = sc[["M1", "M2", "M3"]].all(1)
print(sc.round(2).to_string(index=False))
print(f"\n{len(sc)} concept cells; preferred load-1 trials per cell {sc.pref_trials.min()} to {sc.pref_trials.max()}")
print(f"persistent under (a): {sc.persistent_a.sum()}; not persistent: {(~sc.persistent_a).sum()}")
print(f"persistent under (b): {sc.persistent_b.sum()}; not persistent: {(~sc.persistent_b).sum()}")
print("cells significant in M1, M2, M3:", sc[["M1", "M2", "M3"]].sum().to_dict())
print("by participant (persistent a / cells):", sc.groupby("sub").agg(cells=("nid", "size"), pa=("persistent_a", "sum"), pb=("persistent_b", "sum")).T.to_dict())
for thr in [0.25, 0.75]:
    print(f"threshold variant, persistent when at least {thr:.0%} of bins are higher: {(sc.frac_bins_higher >= thr).sum()} of {len(sc)}")

**Result: reproduced.** 50 of 58 concept cells are not persistent under reading (a) and 48 under
reading (b); with a 25% bin threshold 39 are not persistent.

## fig2c-population-persistence

> Averaged across the 58 concept cells, firing on preferred trials stayed higher than on
> nonpreferred trials across the maintenance epoch of load 1 trials, a pattern of persistent
> activity.

Per cell the trial-mean z-scored, smoothed rate (50 ms bins) on preferred and nonpreferred
trials, in the first encoding second, the first 2.5 s of maintenance and the first second of the
probe (the three panels); per bin a bootstrap over cells (1,000, seed 42) of the mean preferred
minus nonpreferred difference, two-sided, BH over all bins of all panels (per-panel BH as a
variant). Readings: (a) significant bins cover at least half the maintenance bins; (b) every 500 ms
segment holds a significant bin.

In [ ]:
def population_persistence(load, seed=42):
    D = {"enc": [], "maint": [], "probe": []}
    for n in CELLS_C:
        t = T[NT["sub"].iat[n]]
        t = t[t.load == load]
        pm = pref_mask(t, PREF[n], load)
        for ep, on, dur in [("enc", t.enc1.values, 1.0), ("maint", t.maint.values, 2.5), ("probe", t.probe.values, 1.0)]:
            Z = wm.psth_z(SPK[n], on, dur, *FIX[n])
            D[ep].append(np.c_[Z[pm].mean(0), Z[~pm].mean(0)])
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(CELLS_C), (1000, len(CELLS_C)))
    out = {}
    for ep, arr in D.items():
        a = np.array(arr)  # cells x bins x (pref, nonpref)
        d = a[:, :, 0] - a[:, :, 1]
        b = d[idx].mean(1)
        p = np.minimum(1, 2 * np.minimum(np.mean(b <= 0, 0), np.mean(b >= 0, 0)))
        out[ep] = (a.mean(0), d.mean(0), p)
    allp = np.concatenate([out[e][2] for e in ["enc", "maint", "probe"]])
    q = wm.bh(allp)
    nb = {e: len(out[e][2]) for e in out}
    qm = q[nb["enc"]:nb["enc"] + nb["maint"]]
    sig = (qm < 0.05) & (out["maint"][1] > 0)
    sig_panel = (wm.bh(out["maint"][2]) < 0.05) & (out["maint"][1] > 0)
    return out, sig, sig_panel


PERSIST = {}
for load in [1, 2, 3]:
    out, sig, sigp = population_persistence(load)
    PERSIST[load] = (out, sig, sigp)
    seg = [bool(sig[i * 10:(i + 1) * 10].any()) for i in range(5)]
    m = out["maint"][0]
    print(f"load {load}: maintenance mean z preferred {m[:, 0].mean():.3f}, nonpreferred {m[:, 1].mean():.3f}; "
          f"preferred above nonpreferred in {np.mean(m[:, 0] > m[:, 1]):.2f} of bins; significant bins (BH all panels) "
          f"{sig.mean():.2f} (per-panel BH {sigp.mean():.2f}); 500 ms segments with a significant bin {seg}")
    print(f"   ranges: preferred {m[:, 0].min():.2f} to {m[:, 0].max():.2f}, nonpreferred {m[:, 1].min():.2f} to {m[:, 1].max():.2f}")
out = PERSIST[1][0]
fig, ax = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for a, ep, dur in zip(ax, ["enc", "maint", "probe"], [1.0, 2.5, 1.0]):
    x = np.arange(len(out[ep][1])) * 0.05
    a.plot(x, out[ep][0][:, 0], label="preferred")
    a.plot(x, out[ep][0][:, 1], label="nonpreferred")
    a.set_title(f"{ep}, load 1")
    a.set_xlabel("s from onset")
ax[0].set_ylabel("z-scored rate")
ax[0].legend()
plt.show()

**Result: reproduced.** At load 1 the population's preferred trace (mean z 0.761) is above the
nonpreferred one (-0.013) in every maintenance bin, and every bin is significant after BH, so
both readings hold. No point of the plot falls outside its axes, whose ranges are printed.

## ext1bc-persistence-by-load

> The population-level persistence of concept cells was robust at loads 1 and 2 and attenuated at
> load 3.

Same analysis on load 2 and load 3 trials, a trial preferred when any encoded image is the
preferred image; the measure is the fraction of significant maintenance bins.

In [ ]:
for load in [1, 2, 3]:
    out, sig, sigp = PERSIST[load]
    seg = [bool(sig[i * 10:(i + 1) * 10].any()) for i in range(5)]
    print(f"load {load}: fraction of maintenance bins significant {sig.mean():.2f} (per-panel BH {sigp.mean():.2f}); "
          f"meets reading (a) {sig.mean() >= 0.5}, reading (b) {all(seg)}; mean difference {out['maint'][1].mean():.3f} z")

**Result: reproduced.** Loads 1 and 2 have all maintenance bins significant and load 3 has 0.60,
smaller than either, with a smaller preferred minus nonpreferred difference (0.774, 0.487 and
0.325 z).

## fig2d-cross-temporal-localized

> In load 1 trials, decoders trained at specific times of encoding or early maintenance classified
> the image above chance at other times, but cross-time generalization clusters were sparse and
> temporally localized, so the information was not stably held across maintenance.

A pseudo-population of the concept cells on the code's identity-matched trials (load-1 rows of
`new_trial_final.xlsx`, 30 trials, 6 per image; labels the first image), spike counts in 250 ms
windows stepped by 100 ms from first-encoding onset over 5 s (the code's duration for load 1),
z-scored with fixation; a linear SVM trained at each bin and tested at every bin with LOOCV. Per
pair a binomial test against 20%, one-sided (P(X >= k)) or two-sided, BH over the grid. Cluster
rules: (i) the code's, contours of the significance mask longer than 13 points
(`skimage.measure.find_contours`); (ii) the Methods', a cluster-duration null from 1,000 label
shuffles, a cluster kept when its temporal extent (the longer side of its bounding box, in bins,
a choice of this attempt) exceeds the 95th percentile of the largest extent per shuffled map.

In [ ]:
from skimage import measure


def ct_design(nids, load, label):
    subs = sorted(set(NT["sub"].values[nids]))
    rows, _ = wm.pseudo_rows(T, subs, load, "code", FINAL)
    f14 = FINAL[(FINAL.subject_id == 14) & (FINAL.num_images_presented == load)].sort_values("trial_id_final")
    y = f14["stimulus_index_enc1" if label == "first" else f"stimulus_index_enc{load}"].values
    dur = {1: 5, 2: 6, 3: 7}[load]
    st = wm.window_starts(0.25, 0.1, dur)
    X = np.zeros((len(y), len(nids), len(st)))
    for j, n in enumerate(nids):
        sub = NT["sub"].iat[n]
        t = T[sub].set_index("trial").loc[rows[sub]]
        X[:, j, :] = wm.zcounts(SPK[n], t.enc1.values, st, 0.25, *FIX[n])
    # epoch edges relative to first-encoding onset, mean over the participants' trials used
    rel = np.mean([(T[s].set_index("trial").loc[rows[s]].maint - T[s].set_index("trial").loc[rows[s]].enc1).mean() for s in subs])
    return X, y, st, rel


TWO_SIDED = {}


def sig_mask(acc, N, side):
    k = np.round(acc * N).astype(int)
    if side == "one":
        p = binom.sf(k - 1, N, 0.2)
    else:
        p = TWO_SIDED.setdefault(N, np.array([binomtest(kk, N, 0.2).pvalue for kk in range(N + 1)]))[k]
    q = wm.bh(p.ravel()).reshape(p.shape)
    return (q < 0.05) & (acc > 0.2)


def clusters(mask):
    lab, n = ndimage.label(mask)
    out = []
    for i, sl in enumerate(ndimage.find_objects(lab)):
        ext = max(sl[0].stop - sl[0].start, sl[1].stop - sl[1].start)
        out.append((i + 1, ext))
    return lab, out


def code_rule(mask):
    keep = np.zeros_like(mask)
    for c in measure.find_contours(mask.astype(float), 0.5):
        if len(c) > 13:
            r0, r1 = int(np.floor(c[:, 0].min())), int(np.ceil(c[:, 0].max()))
            c0, c1 = int(np.floor(c[:, 1].min())), int(np.ceil(c[:, 1].max()))
            keep[r0:r1 + 1, c0:c1 + 1] |= mask[r0:r1 + 1, c0:c1 + 1]
    return keep


def ct_analysis(nids, load, label, n_perm=1000, seed=0):
    X, y, st, rel = ct_design(nids, load, label)
    acc = wm.loo_cross_temporal(X, y)
    seeds = [wm.stable_seed("ct", load, label, len(nids), seed, i) for i in range(n_perm)]
    chunks = [seeds[i::4] for i in range(4)]
    perm_maps = sum(wm.pmap(wm.ct_perm_maps, [(X, y, c) for c in chunks]), [])
    res = {"acc": acc, "st": st, "rel": rel, "N": len(y)}
    for side in ["one", "two"]:
        m = sig_mask(acc, len(y), side)
        null_ext = []
        for pm in perm_maps:
            _, cl = clusters(sig_mask(pm, len(y), side))
            null_ext.append(max([e for _, e in cl], default=0))
        thr = np.percentile(null_ext, 95)
        lab, cl = clusters(m)
        keep_perm = np.isin(lab, [i for i, e in cl if e > thr])
        res[side] = {"mask": m, "null95": thr, "perm": keep_perm, "code": code_rule(m),
                     "null_ext_mean": float(np.mean(null_ext))}
    return res


def summarize_ct(res, name):
    st, rel, acc = res["st"], res["rel"], res["acc"]
    enc_b = st + 0.25 <= rel
    mb = (st >= rel) & (st + 0.25 <= rel + 2.5)
    early = (st >= rel) & (st + 0.25 <= rel + 1.0)
    offdiag = ~np.eye(len(st), dtype=bool)
    rows = []
    for side in ["one", "two"]:
        for rule in ["code", "perm"]:
            k = res[side][rule]
            rows.append({"map": name, "binomial": side, "cluster_rule": rule,
                         "sig_pairs": int(res[side]["mask"].sum()), "kept_pairs": int(k.sum()),
                         "kept_offdiag": int((k & offdiag).sum()),
                         "train_bins_enc_or_early_with_offdiag": int(((k & offdiag)[enc_b | early]).any(1).sum()),
                         "max_frac_maint_test_bins_from_a_maint_train_bin": round(float(k[np.ix_(mb, mb)].mean(1).max()) if mb.any() else 0, 2),
                         "null95_extent": res[side]["null95"]})
    return pd.DataFrame(rows)


CT = {}
CT[("all", 1, "first")] = ct_analysis(CELLS_C, 1, "first")
r = CT[("all", 1, "first")]
print(f"{len(CELLS_C)} concept cells, {r['N']} trials, {len(r['st'])} bins; maintenance starts {r['rel']:.2f} s after first-encoding onset")
print(f"accuracy: diagonal mean {np.diag(r['acc']).mean():.3f}, range {r['acc'].min():.3f} to {r['acc'].max():.3f}; "
      f"mean largest cluster extent in shuffled maps {r['one']['null_ext_mean']:.1f} bins")
print(summarize_ct(r, "load 1, all concept cells").to_string(index=False))
im = plt.imshow(r["acc"] * 100, origin="lower", vmin=0, vmax=100, cmap="viridis")
if r["one"]["code"].any():
    plt.contour(r["one"]["code"].astype(float), levels=[0.5], colors="r")
plt.xlabel("test bin")
plt.ylabel("train bin")
plt.colorbar(im, label="accuracy (%)")
plt.title("load 1 cross-temporal decoding (red: kept by the code's rule, one-sided)")
plt.show()

**Result: reproduced.** Under every reading, 17 training bins in encoding or early maintenance
have off-diagonal pairs kept above chance, and no maintenance training bin generalizes to more
than 0.18 of the maintenance test bins, so the clusters are localized. The shuffled maps had no
supra-threshold cluster (their 95th-percentile extent is 0), so the Methods' cluster rule removes
nothing here.

## ext1de-cross-temporal-load23

> In load 2 and load 3 trials cross-time generalization was minimal.

The analysis of fig2d on load 2 (6 s) and load 3 (7 s) rows of the identity-matched table,
labels the first image (Methods) or the last (the code); "minimal" read as fewer off-diagonal
pairs in surviving clusters than at load 1 under the same rule.

In [ ]:
for load in [2, 3]:
    for label in ["first", "last"]:
        CT[("all", load, label)] = ct_analysis(CELLS_C, load, label)
tabs = [summarize_ct(CT[("all", 1, "first")], "load 1")]
for load in [2, 3]:
    for label in ["first", "last"]:
        tabs.append(summarize_ct(CT[("all", load, label)], f"load {load}, label {label}"))
ct_tab = pd.concat(tabs)
print(ct_tab.to_string(index=False))
base = ct_tab[ct_tab["map"] == "load 1"].set_index(["binomial", "cluster_rule"]).kept_offdiag
for label in ["first", "last"]:
    for (side, rule), b in base.items():
        v = [int(ct_tab[(ct_tab["map"] == f"load {L}, label {label}") & (ct_tab.binomial == side) & (ct_tab.cluster_rule == rule)].kept_offdiag.iloc[0]) for L in [2, 3]]
        print(f"label {label}, {side}-sided, rule {rule}: off-diagonal kept pairs load 1 {b}, load 2 {v[0]}, load 3 {v[1]}; both fewer: {v[0] < b and v[1] < b}")

**Result: reproduced.** Under both labels and every rule, loads 2 (6 to 10 off-diagonal pairs)
and 3 (none) keep far fewer off-diagonal pairs than load 1 (217 to 280).

## fig2e-maintenance-decoding-amy-mtl

> Concept cells in the amygdala and in the MTL carried significant stimulus information during
> load 1 maintenance (35.56% and 32.13% decoding accuracy, chance 20%).

Within each participant, its concept cells in the region group, all its trials of the load,
labels the last image; z-scored counts in 1 s windows stepped by 250 ms over the first 2.5 s of
maintenance, concatenated; linear SVM, LOOCV; accuracy the mean over participants; p from 1,000
label shuffles, (k + 1) / 1001 on the mean over participants, BH within the load over the six
region groups of the code (main_2e,supp_1f.py:14-21). Participants need at least three (Methods)
or more than three (the code) concept cells in the group.

In [ ]:
GROUPS_WP = {"AMY": ["AMY"], "HPC": ["HPC"], "dACC": ["dACC"], "preSMA": ["preSMA"],
             "MTL": ["AMY", "HPC"], "dACC_preSMA": ["dACC", "preSMA"]}
ST_WP = wm.window_starts(1.0, 0.25, 2.5)


def wp_design(g, sub, load):
    nids = NT[(NT["sub"] == sub) & NT.region.isin(GROUPS_WP[g]) & CONCEPT[NT.nid]].nid.values
    t = T[sub][T[sub].load == load]
    X = np.concatenate([wm.zcounts(SPK[n], t.maint.values, ST_WP, 1.0, *FIX[n]) for n in nids], 1)
    return X @ X.T, t["last"].values


units, designs = [], []
for g, regs in GROUPS_WP.items():
    for sub in sorted(T):
        k = int((CONCEPT & (NT["sub"] == sub).values & NT.region.isin(regs).values).sum())
        if k >= 3:
            for load in [1, 2, 3]:
                units.append((g, sub, load, k))
                designs.append(wp_design(g, sub, load))
nulls = wm.pmap(wm.perm_null_loo, [(K, y, 1000, wm.stable_seed("wp", g, s, L)) for (g, s, L, _), (K, y) in zip(units, designs)])
WP = pd.DataFrame([{"group": g, "sub": s, "load": L, "cells": k, "acc": wm.loo_correct(K, y).mean(), "null": nl}
                   for (g, s, L, k), (K, y), nl in zip(units, designs, nulls)])
print(WP[WP.load == 1][["group", "sub", "cells", "acc"]].assign(acc=lambda d: (100 * d.acc).round(2)).to_string(index=False))


def wp_stats(min_cells, load):
    rows = []
    for g in GROUPS_WP:
        x = WP[(WP.group == g) & (WP.load == load) & (WP.cells >= min_cells)]
        if len(x) == 0:
            continue
        m = x.acc.mean()
        nm = np.mean(np.stack(x.null.values), 0)
        rows.append({"group": g, "participants": len(x), "accuracy_%": round(100 * m, 2),
                     "null_mean_%": round(100 * nm.mean(), 2), "p": (np.sum(nm >= m) + 1) / 1001})
    df = pd.DataFrame(rows)
    df["p_BH"] = wm.bh(df.p)
    return df


for mc, name in [(3, "at least three cells (Methods)"), (4, "more than three cells (the code)")]:
    print(f"\nload 1, {name}:")
    print(wp_stats(mc, 1).to_string(index=False))
print("\npaper: AMY 35.56% (P = 0.001), MTL 32.13% (P = 0.001); tolerance 32.0 to 39.1 and 28.9 to 35.3")

**Result: reproduced.** At load 1 the amygdala decodes at 35.63% and the MTL at 33.61% with at
least three cells (p = 0.001 for both), and at 34.33% and 32.89% with more than three cells;
all are within tolerance.

## ext1f-maintenance-decoding-load2

> Amygdala and MTL concept cells also carried significant information about the last image during
> load 2 maintenance (23.56% and 28.89%).

In [ ]:
for mc, name in [(3, "at least three cells (Methods)"), (4, "more than three cells (the code)")]:
    print(f"load 2, {name}:")
    print(wp_stats(mc, 2).to_string(index=False))
print("\npaper: AMY 23.56% (P = 0.031), MTL 28.89% (P = 0.002)")

**Result: partially reproduced.** At load 2 both groups decode above the null (p = 0.002), and
the MTL accuracy is within tolerance (26.67% and 29.56% under the two rules), but the amygdala
accuracy (26.43% and 27.89%) is above its tolerance of 21.2 to 25.9%.

## fig2f-bin-width-decoding

> In load 1 trials, pseudo-population decoding of the image during maintenance was above chance at
> every bin width and rose with bin width (250 ms 40.00%, 500 ms 56.67%, 750 ms 66.67%, 1000 ms
> 80.00%).

All concept cells, z-scored counts in windows of each width slid by 25 ms over the first 2.5 s of
maintenance, concatenated; linear SVM, LOOCV; p from 1,000 label shuffles, (k + 1) / 1001; BH
over the 12 tests of 4 widths and 3 loads. Trial readings: (a) the code's identity-matched rows
(`new_trial_final.xlsx`), labels as the code takes them (subject 14's); (b) each participant's
first six trials of each label image, aligned by image and rank. The fast SVM used here is
checked against sklearn's `SVC(kernel='linear')`.

In [ ]:
WIDTHS = [0.25, 0.5, 0.75, 1.0]


def bw_design(nids, load, reading, width):
    subs = sorted(set(NT["sub"].values[nids]))
    rows, lab = wm.pseudo_rows(T, subs, load, reading, FINAL)
    st = wm.window_starts(width, 0.025, 2.5)
    B = []
    for n in nids:
        sub = NT["sub"].iat[n]
        t = T[sub].set_index("trial").loc[rows[sub]]
        B.append(wm.zcounts(SPK[n], t.maint.values, st, width, *FIX[n]))
    return np.concatenate(B, 1), np.asarray(lab)


for reading in ["code", "rank"]:
    X, y = bw_design(CELLS_C, 1, reading, 1.0)
    print(f"reading {reading}: fast SVM equals sklearn SVC(kernel='linear') in every LOOCV prediction:",
          bool((wm.loo_correct(X @ X.T, y) == wm.loo_correct_sklearn(X, y)).all()))


def bw_run(name, nids):
    keys = [(reading, load, w) for reading in ["code", "rank"] for load in [1, 2, 3] for w in WIDTHS]
    des = [bw_design(nids, L, r, w) for r, L, w in keys]
    Ks = [X @ X.T for X, _ in des]
    nulls = wm.pmap(wm.perm_null_loo, [(K, y, 1000, wm.stable_seed("bw", name, *k)) for k, K, (_, y) in zip(keys, Ks, des)])
    res = {k: (wm.loo_correct(K, y), nl, X.shape) for k, K, (X, y), nl in zip(keys, Ks, des, nulls)}
    rows = []
    for (reading, load, w), (real, null, shape) in res.items():
        rows.append({"reading": reading, "load": load, "width_ms": int(w * 1000), "trials": shape[0], "features": shape[1],
                     "accuracy_%": round(100 * real.mean(), 2), "null_mean_%": round(100 * null.mean(), 2),
                     "p": (np.sum(null >= real.mean()) + 1) / 1001})
    df = pd.DataFrame(rows)
    df["p_BH"] = np.nan
    for reading in ["code", "rank"]:
        m = df.reading == reading
        df.loc[m, "p_BH"] = wm.bh(df.p[m])
    return res, df


BW = {}
BW["all"] = bw_run("all", CELLS_C)
print(BW["all"][1].to_string(index=False))
print("\npaper, load 1: 40.00, 56.67, 66.67, 80.00% (tolerance 10% of each), Pmax = 0.001")

In [ ]:
# Variant: how the decoding depends on which six trials of each image enter. 20 random draws of
# six load-1 trials per image in each participant (aligned by image and draw rank), 1000 ms width.
rng = np.random.default_rng(23)
subs = sorted(set(NT["sub"].values[CELLS_C]))
st = wm.window_starts(1.0, 0.025, 2.5)
accs = []
for d in range(20):
    rows = {}
    for s in subs:
        t = T[s][T[s].load == 1]
        rows[s] = np.concatenate([rng.permutation(t[t.img1 == im].trial.values)[:6] for im in range(1, 6)])
    B = []
    for n in CELLS_C:
        sub = NT["sub"].iat[n]
        t = T[sub].set_index("trial").loc[rows[sub]]
        B.append(wm.zcounts(SPK[n], t.maint.values, st, 1.0, *FIX[n]))
    Xr = np.concatenate(B, 1)
    accs.append(wm.loo_correct(Xr @ Xr.T, np.repeat(np.arange(1, 6), 6)).mean())
accs = 100 * np.array(accs)
print(f"random six-per-image draws, load 1, 1000 ms: accuracy median {np.median(accs):.1f}%, range {accs.min():.1f} to {accs.max():.1f}%")
real_code = 100 * BW["all"][0][("code", 1, 1.0)][0].mean()
print(f"the code's table gives {real_code:.2f}%; draws at or above it: {(accs >= real_code).sum()} of 20")

**Result: partially reproduced.** Under both trial readings load-1 accuracy is above chance at
every width (BH p at most 0.009) and rises from 250 to 1000 ms, but not every value is within
tolerance: reading (a) gives 30.00, 60.00, 66.67 and 76.67% (250 ms below its tolerance of 36 to
44%), reading (b) 50.00, 66.67, 73.33 and 76.67% (250 and 500 ms above theirs). Random draws of
six trials per image give 56.7 to 86.7% at 1000 ms, so the values depend on which trials enter.

## fig2f-bin-width-pairwise

> Pairwise, accuracy was higher for 750 than 250 ms, for 1000 than 250 ms and for 1000 than 500 ms
> bins, while 500 against 250, 1000 against 750 and 750 against 500 ms did not differ after
> correction.

Readings: (a) a paired permutation over trials swapping each trial's LOOCV correctness between the
two widths (1,000), two-sided; (b) the observed difference against the differences of the two
widths' shuffled nulls (paired by shuffle index), two-sided; BH over the six pairs.

In [ ]:
import itertools
rng = np.random.default_rng(31)
PAIRS = [(0.75, 0.25), (1.0, 0.25), (1.0, 0.5), (0.5, 0.25), (1.0, 0.75), (0.75, 0.5)]
for reading in ["code", "rank"]:
    res = BW["all"][0]
    rows = []
    for a, b in PAIRS:
        ca, na, _ = res[(reading, 1, a)]
        cb, nb, _ = res[(reading, 1, b)]
        obs = ca.mean() - cb.mean()
        sw = rng.random((1000, len(ca))) < 0.5
        nulla = (np.where(sw, cb, ca).mean(1) - np.where(sw, ca, cb).mean(1))
        pa = (np.sum(np.abs(nulla) >= abs(obs)) + 1) / 1001
        dn = na - nb
        pb = (np.sum(np.abs(dn - dn.mean()) >= abs(obs)) + 1) / 1001
        rows.append({"pair_ms": f"{int(a * 1000)} vs {int(b * 1000)}", "diff_%": round(100 * obs, 2), "p_a": pa, "p_b": pb})
    df = pd.DataFrame(rows)
    df["p_a_BH"] = wm.bh(df.p_a)
    df["p_b_BH"] = wm.bh(df.p_b)
    print(f"trial reading {reading}:")
    print(df.round(4).to_string(index=False))
print("paper: 750 vs 250 P = 0.026, 1000 vs 250 P = 0.004, 1000 vs 500 P = 0.032; the other three P > 0.05")

**Result: not reproduced.** Under neither reading are exactly the three named pairs significant:
with the code's table 750 vs 250 and 1000 vs 250 are significant but 1000 vs 500 is not (BH p
0.079 and 0.139) and 500 vs 250 is (0.010 and 0.002); with reading (b) 750 vs 250 and 1000 vs 250
are significant (BH p 0.036 to 0.048), the three pairs the paper calls not significant are not,
and 1000 vs 500 is not significant (0.363 and 0.483).

## ext1g-bin-width-load2

> Bin width had little effect on decoding of the second image in load 2 trials, and load 2
> accuracy was lower than load 1 accuracy.

In [ ]:
df = BW["all"][1]
for reading in ["code", "rank"]:
    l1 = df[(df.reading == reading) & (df.load == 1)].set_index("width_ms")["accuracy_%"]
    l2 = df[(df.reading == reading) & (df.load == 2)].set_index("width_ms")["accuracy_%"]
    print(f"reading {reading}: load 2 accuracies {l2.to_dict()}; range {l2.max() - l2.min():.2f} points "
          f"(criterion: below half the load 1 range of 40.0, i.e. 20.0); below load 1 at every width: {bool((l2 < l1).all())}")

**Result: reproduced.** The load 2 accuracies span 16.0 points with the code's table and 4.0
points with reading (b), below the 20-point threshold, and each is below the load 1 accuracy at
the same width.

## fig3abc-trajectory-complexity, fig3d-encoding-vs-maintenance-separability, fig3h-early-vs-late-maintenance (blocked)

These claims rest on 100 stratified train/test splits, each with its own dPCA fit (Kobak's
package, regularization chosen by its built-in cross-validation, as `opt_regularizer_flag = True`
in 03_dpca/main_3d,supp_2a,b,i.py:119-131) for encoding (1 s) and maintenance (2.5 s) in 2 ms
bins, and for the early and late halves of maintenance. The cell below times one split of the
claim's design on the data (58 concept cells, 3 training trials per image).

In [ ]:
from dPCA.dPCA import dPCA
from scipy.ndimage import gaussian_filter1d

subs = sorted(set(NT["sub"].values[CELLS_C]))
rows, lab = wm.pseudo_rows(T, subs, 1, "code", FINAL)
rng = np.random.default_rng(3)
train = np.concatenate([rng.permutation(np.where(lab == c)[0])[:3] for c in np.unique(lab)])


def dpca_tensor(onset_col, dur):
    nb = int(round(dur / 0.002))
    out = np.zeros((3, len(CELLS_C), 5, nb))
    for j, n in enumerate(CELLS_C):
        sub = NT["sub"].iat[n]
        t = T[sub].set_index("trial").loc[rows[sub]]
        mu, sd = FIX[n]
        for ci, c in enumerate(np.unique(lab)):
            idx = [i for i in train if lab[i] == c]
            for r_, i in enumerate(idx):
                on = t[onset_col].values[i]
                e = on + np.arange(nb + 1) * 0.002
                cnt = np.diff(np.searchsorted(SPK[n], e)) / 0.002
                z = (cnt - mu) / sd if sd > 0 else cnt - mu
                out[r_, j, ci] = gaussian_filter1d(z, 60)
    return out


timing = {}
for name, col, dur in [("encoding", "enc1", 1.0), ("maintenance", "maint", 2.5)]:
    trialX = dpca_tensor(col, dur)
    Xm = trialX.mean(0)
    Xm = Xm - Xm.reshape(Xm.shape[0], -1).mean(1)[:, None, None]
    t0 = time.time()
    d = dPCA(labels="st", regularizer="auto")
    d.opt_regularizer_flag = True
    d.protect = []
    d.fit_transform(Xm, trialX)
    timing[name] = time.time() - t0
print({k: round(v, 1) for k, v in timing.items()}, "s for one split")
per_split = timing["encoding"] + timing["maintenance"]
print(f"fig3d needs 100 splits: about {100 * per_split / 60:.0f} min; fig3h adds two half-maintenance fits per split, "
      f"about {100 * timing['maintenance'] / 60:.0f} min more (each half is half the bins); "
      f"together about {100 * (per_split + timing['maintenance']) / 60:.0f} min, over the 60 min limit for a notebook")

**Result: blocked (too large).** One split takes about 45 to 50 s (printed above: encoding plus
maintenance), so the 100 splits of fig3d take well over an hour, beyond the sixty-minute limit for
a notebook, and fig3h (about 50 min more) and fig3abc (which uses the same fits) add to it. The
three claims are not tested.

## acg-fit-yield

> Fitting CellExplorer's triple-exponential model to each neuron's autocorrelogram and keeping
> well-fit units (R2 > 0.3) left 294 neurons for cell-type analysis.

A port of the code's MATLAB functions (calc_ACG_metrics.m, fit_ACG.m, Mean_ACG.m,
normalize_acg_to_unit.m): narrow ACG (+/- 50 ms, 0.5 ms bins), the triple exponential fitted with
CellExplorer's start point and bounds by bounded nonlinear least squares (scipy's trust-region
reflective method in place of MATLAB's `fit`). Readings: (a) every spike of the unit; (b) only
spikes within the trials (fixation onset to response).

In [ ]:
FEAT = {}
for reading in ["a", "b"]:
    rows = []
    for n in range(len(NT)):
        s = SPK[n]
        if reading == "b":
            t = T[NT["sub"].iat[n]]
            keep = np.zeros(len(s), bool)
            for a_, b_ in zip(t.fix_start.values, t.timestamps_Response.values):
                i, j = np.searchsorted(s, [a_, b_])
                keep[i:j] = True
            f = wm.acg_features(s[keep])
            f["firing_rate"] = len(s) / (s[-1] - s[0])
        else:
            f = wm.acg_features(s)
        rows.append(f)
    FEAT[reading] = pd.DataFrame(rows)
for reading in ["a", "b"]:
    F = FEAT[reading]
    print(f"reading ({reading}): R2 > 0.3: {(F.r2 > 0.3).sum()}, R2 >= 0.3: {(F.r2 >= 0.3).sum()}, "
          f"fits with no R2 (too few spikes): {F.r2.isna().sum()}; R2 median {F.r2.median():.3f}")
F = FEAT["a"]
cmp = pd.DataFrame({
    "feature": ["firing_rate", "acg_norm", "burst_royer", "cv_isi", "r2", "tau_rise", "tau_decay"],
    "shipped_column": ["firing_rate", "acg_norm", "Burst", "CV2", "R2", "tau_rise", "Decay"]})
cmp["max_abs_diff"] = [np.nanmax(np.abs(F[a].values - SHIP_CM[b].values)) for a, b in zip(cmp.feature, cmp.shipped_column)]
cmp["frac_within_1pct"] = [np.nanmean(np.abs(F[a].values - SHIP_CM[b].values) <= 0.01 * np.abs(SHIP_CM[b].values) + 1e-9)
                           for a, b in zip(cmp.feature, cmp.shipped_column)]
print("\nfor comparison, reading (a) against the code's shipped Cell_metrics.xlsx:")
print(cmp.round(4).to_string(index=False))
print(f"shipped table: R2 > 0.3 for {(SHIP_CM.R2 > 0.3).sum()}, R2 >= 0.3 for {(SHIP_CM.R2 >= 0.3).sum()}; "
      f"agreement on R2 > 0.3 with reading (a): {np.mean((F.r2 > 0.3) == (SHIP_CM.R2 > 0.3)):.3f}")
print("paper: 294 (tolerance 264.6 to 323.4)")

**Result: reproduced.** 317 neurons have R2 > 0.3 under reading (a) and 298 under reading (b),
both within tolerance. The firing rate and ISI CV of reading (a) equal the code's shipped table
exactly, and the normalized mean ACG and Royer burst index agree within 1% for 99% of neurons, so
the code used every spike of the unit; the fitted R2 agrees within 1% for 48% of neurons and the
time constants for 7 to 11%, as the optimizer differs from MATLAB's. Classification at R2 > 0.3
agrees for 97.5% of neurons.

## fig5c-cell-type-clusters

> Spectral clustering on firing rate, mean normalised ACG and ACG rise time constant split the 294
> neurons into two clusters, 92 putative pyramidal neurons and 202 putative interneurons.

In [ ]:
TYPES = {}
for reading in ["a", "b"]:
    F = FEAT[reading]
    d = F[F.r2 > 0.3].dropna(subset=["firing_rate", "acg_norm", "tau_rise"])
    ty, lab = wm.spectral_types(d)
    types = np.array([None] * len(NT), dtype=object)
    types[d.index.values] = ty
    TYPES[reading] = types
    print(f"reading ({reading}): {len(d)} neurons; PY {(ty == 'PY').sum()}, IN {(ty == 'IN').sum()}; "
          f"medians PY / IN: firing rate {d.firing_rate[ty == 'PY'].median():.2f} / {d.firing_rate[ty == 'IN'].median():.2f} Hz, "
          f"acg_norm {d.acg_norm[ty == 'PY'].median():.3f} / {d.acg_norm[ty == 'IN'].median():.3f}, "
          f"tau_rise {d.tau_rise[ty == 'PY'].median():.2f} / {d.tau_rise[ty == 'IN'].median():.2f} ms")
d = SHIP_CM[SHIP_CM.R2 >= 0.3].dropna(subset=["firing_rate", "acg_norm", "tau_rise"])
ty, lab = wm.spectral_types(d)
print(f"for comparison, the same clustering on the code's shipped features (R2 >= 0.3, {len(d)} neurons): PY {(ty == 'PY').sum()}, IN {(ty == 'IN').sum()}; "
      f"the code's own rule (cluster label 1 is PY) gives PY {(lab == 1).sum()}")
TY = TYPES["a"]
print("paper: PY 92 (82.8 to 101.2), IN 202 (181.8 to 222.2)")

**Result: reproduced.** Under reading (b) the counts are PY 100 and IN 198, both within
tolerance; under reading (a) IN (215) is within tolerance and PY (102) just above it. The
clustering applied to the code's shipped features gives exactly 92 and 202, so the difference
comes from the ACG fit. Later sections use the types of reading (a), the spikes the code used
(see acg-fit-yield), with reading (b) shown where it bears on a value.

## fig5dfh-cluster-features

> Putative interneurons had higher firing rates, greater normalised mean ACG amplitude and larger
> tau_rise than putative pyramidal neurons.

In [ ]:
def type_tests(F, types, cols):
    rows = []
    for c in cols:
        a, b = F[c][types == "IN"].dropna(), F[c][types == "PY"].dropna()
        u = mannwhitneyu(a, b, alternative="two-sided")
        rows.append({"feature": c, "IN_median": a.median(), "PY_median": b.median(), "U_IN_vs_PY": u.statistic, "p": u.pvalue})
    df = pd.DataFrame(rows)
    from statsmodels.stats.multitest import multipletests
    df["p_BY"] = multipletests(df.p, method="fdr_by")[1]
    return df


for reading in ["a", "b"]:
    df = type_tests(FEAT[reading], TYPES[reading], ["firing_rate", "acg_norm", "tau_rise", "burst_royer", "cv_isi"])
    print(f"reading ({reading}):")
    print(df.to_string(index=False))
print("paper: U = 17294, 18522, 13465 (IN vs PY), P = 1.42e-32, 2.16e-41, 2.58e-9")

**Result: reproduced.** IN are higher on all three features under both readings, BY-corrected p
at most 8e-05 (U = 20822, 21885 and 14159 under reading a); as the claims file notes, this follows
from the clustering by construction.

## fig5gh-burst-index-cv2

> Putative pyramidal neurons had a higher burst index (0.985 against 0.622) and greater
> spike-train irregularity (CV2 1.673 against 1.319) than putative interneurons.

In [ ]:
for reading in ["a", "b"]:
    F, ty = FEAT[reading], TYPES[reading]
    df = type_tests(F, ty, ["firing_rate", "acg_norm", "tau_rise", "burst_royer", "cv_isi"])
    dh = type_tests(F, ty, ["firing_rate", "acg_norm", "tau_rise", "burst_royer", "cv2_holt"])
    print(f"reading ({reading}):")
    for c, tab in [("burst_royer", df), ("cv_isi", df), ("cv2_holt", dh)]:
        r = tab[tab.feature == c].iloc[0]
        a, b = F[c][ty == "PY"], F[c][ty == "IN"]
        print(f"  {c}: PY mean {a.mean():.3f} median {a.median():.3f}; IN mean {b.mean():.3f} median {b.median():.3f}; "
              f"U(IN vs PY) = {r.U_IN_vs_PY:.0f}, BY-corrected p = {r.p_BY:.2e}")
# The claim takes CV2 from the session's interspike intervals, whichever spikes the ACG used:
# types and burst index of reading (b) with the ISI measures of all of each unit's spikes.
F, ty = FEAT["b"], TYPES["b"]
Fs = FEAT["a"]
from statsmodels.stats.multitest import multipletests
ps = [mannwhitneyu(F.burst_royer[ty == "IN"], F.burst_royer[ty == "PY"]).pvalue,
      mannwhitneyu(Fs.cv_isi[ty == "IN"], Fs.cv_isi[ty == "PY"]).pvalue,
      mannwhitneyu(Fs.cv2_holt[ty == "IN"], Fs.cv2_holt[ty == "PY"]).pvalue]
print("reading (b) types, session ISIs:")
for c, src, p in [("burst_royer", F, ps[0]), ("cv_isi", Fs, ps[1]), ("cv2_holt", Fs, ps[2])]:
    a, b = src[c][ty == "PY"], src[c][ty == "IN"]
    print(f"  {c}: PY mean {a.mean():.3f} median {a.median():.3f}; IN mean {b.mean():.3f} median {b.median():.3f}; p = {p:.2e}")
print("paper: burst index PY 0.985 vs IN 0.622, CV2 PY 1.673 vs IN 1.319; tolerance 10% of each")

**Result: reproduced.** With the types and burst index of reading (b) and the CV of each unit's
session interspike intervals (the column the code feeds to Fig. 5h), the medians are burst index
PY 0.947 and IN 0.622 and CV PY 1.593 and IN 1.322, all within tolerance, PY higher on both with
p below 0.005 (printed above). The group means of the
burst index are far from the paper's (PY 5.6, a few very large values), so the paper's numbers
match medians. Under reading (a) the PY burst-index median (0.802) is outside tolerance and its
corrected p is 0.11. Holt's CV2 is lower than the paper's values (PY 1.18, IN 0.93).

## supp4-rodent-validation (data elsewhere)

> Applied to a labelled rodent dataset of 44 pyramidal neurons and 299 PV and SST interneurons, the
> spectral-clustering classifier assigned 333 of 343 neurons correctly (97%).

Tested on the features of the CellExplorer ground-truth cells that the paper's code ships
(`validation_data.xlsx`), clustered as in fig5c, each cluster labelled by its majority type.

In [ ]:
v = VALID.copy()
print("cell types in the table:", v.cell_type.value_counts().to_dict(), "; rows:", len(v))
vv = v.rename(columns={"norm_acg": "acg_norm"}).dropna(subset=["firing_rate", "acg_norm", "tau_rise", "tau_decay", "cell_type"])
_, lab = wm.spectral_types(vv)
vv["cluster"] = lab
maj = vv.groupby("cluster").cell_type.agg(lambda s: s.mode().iat[0])
vv["match"] = vv.cell_type == vv.cluster.map(maj)
print(pd.crosstab(vv.cluster, vv.cell_type).to_string())
print(f"matched {vv.match.sum()} of {len(vv)}; misassigned by type: {vv[~vv.match].cell_type.value_counts().to_dict()}")
print("paper: 333 of 343, one pyramidal and nine interneurons misassigned")

**Result: reproduced.** The clustering assigns 333 of the 343 ground-truth cells to their
cluster's majority type, with one pyramidal neuron and nine interneurons misassigned, as the
paper reports. These are the features the paper's code ships, not the CellExplorer recordings.

## fig4c-encoding-bursts-vs-poisson

> During encoding, population bursting did not exceed a rate-matched Poisson null for most neuronal
> groups; it differed from the null only for all neurons and for the low mean ACG amplitude group.

Within each participant and group, load 1 trials, the first encoding second; pooled spikes of the
group, trials concatenated, 70 ms bins, Gaussian smoothing (sigma 40 ms), peaks at least 2 bins
apart; 100 rate-matched Poisson surrogates; paired two-sided Wilcoxon signed-rank over
participants (the code's `zero_method='pratt'`, asymptotic) and BH over the code's eight groups.
Readings: peak rule (prominence, the code and Results; or height above the 90th percentile, the
Methods), surrogate threshold (the real train's, the code; or each surrogate's own), group rule
(the code's 6 lowest and highest; or the Methods' median split), all neurons (902 in participants
with at least 10, or the well-fit neurons), and a 1-bin spacing variant.

In [ ]:
EP = {}
for n in range(len(NT)):
    t_all = T[NT["sub"].iat[n]]
    for L in [1, 2, 3]:
        t = t_all[t_all.load == L]
        enc_on = {1: t.enc1, 2: t.timestamps_Encoding2, 3: t.timestamps_Encoding3}[L].values
        EP[(n, L, "enc")] = wm.epoch_spikes(SPK[n], enc_on, enc_on + 1.0, 1.0)
        EP[(n, L, "maint28")] = wm.epoch_spikes(SPK[n], t.maint.values, t.probe.values, 2.8)
        EP[(n, L, "maint3")] = wm.epoch_spikes(SPK[n], t.maint.values, t.probe.values, 3.0)
        EP[(n, L, "probe")] = wm.epoch_spikes(SPK[n], t.probe.values, t.probe.values + 1.0, 1.0)
GR = {g: wm.burst_groups(NT, FEAT["a"], CONCEPT, TY, g) for g in ["code", "methods"]}
for g, G in GR.items():
    print(f"group rule {g}:", {k: f"{len(v)} participants, {sum(len(x) for x in v.values())} neurons" for k, v in G.items()})

FAM = ["All902", "LowACG", "HighACG", "ShortDecay", "LongDecay", "Concept", "PY", "IN"]
CONFIGS = [("prominence", 2, "real"), ("height", 2, "real"), ("prominence", 2, "own"), ("height", 2, "own"),
           ("prominence", 1, "real"), ("height", 1, "real")]
rows = []
for grule, G in GR.items():
    for rule, dist, thr in CONFIGS:
        for epoch, key, win in [("enc", "enc", 1.0), ("maint", "maint28", 2.8)]:
            for g, subs in G.items():
                for sub, nids in subs.items():
                    rng = np.random.default_rng(wm.stable_seed(g, sub, epoch, rule, dist, thr, grule))
                    rc, scount, _, _ = wm.group_bursts([EP[(n, 1, key)] for n in nids], win, rng, rule, dist, thr)
                    rows.append(dict(grule=grule, rule=rule, dist=dist, thr=thr, epoch=epoch, group=g, sub=sub, real=rc, sur=scount))
BR = pd.DataFrame(rows)


def real_vs_null(df, groups=FAM, extra=("AllFit",)):
    out = []
    for g in list(groups) + list(extra):
        x = df[df.group == g]
        if len(x) < 2:
            continue
        w = wilcoxon(x.real, x.sur, zero_method="pratt", method="asymptotic")
        out.append({"group": g, "n": len(x), "real_median": x.real.median(), "null_median": round(x.sur.median(), 1),
                    "real_above": int((x.real > x.sur).sum()), "W": w.statistic, "p": w.pvalue})
    o = pd.DataFrame(out)
    m = o.group.isin(groups)
    o.loc[m, "p_BH"] = wm.bh(o.p[m])
    o.loc[~m, "p_BH"] = np.nan
    return o


def show_bursts(epoch):
    summ = []
    for (grule, rule, dist, thr), df in BR[BR.epoch == epoch].groupby(["grule", "rule", "dist", "thr"]):
        o = real_vs_null(df)
        o.insert(0, "reading", f"{grule}/{rule}/d{dist}/{thr}")
        summ.append(o)
    return pd.concat(summ)


BURST_ENC = show_bursts("enc")
print(BURST_ENC.round(4).to_string(index=False))

In [ ]:
def fig4c_rule(o, all_name="All902"):
    sig = dict(zip(o.group, o.p_BH < 0.05))
    if all_name == "AllFit":
        sig["All902"] = bool(o[o.group == "AllFit"].p.iloc[0] < 0.05)
    named = sig["All902"] and sig["LowACG"]
    others = [g for g in ["HighACG", "ShortDecay", "LongDecay", "Concept"] if sig[g]]
    return named, others


for reading, o in BURST_ENC.groupby("reading", sort=False):
    for alln in ["All902", "AllFit"]:
        named, others = fig4c_rule(o, alln)
        print(f"{reading}, all neurons as {alln}: all and low ACG differ {named}; other groups that differ {others}")

**Result: reproduced.** Under the Methods' groups with the height rule and the real threshold
(spacing 2 or 1 bin), all neurons (BH p 0.0054) and the low ACG group (0.0483) differ from the
Poisson null and the other four groups do not. Under the code's groups with the prominence rule
the same two differ, but the concept group does too (BH p 0.045), and the Methods' groups with
prominence make every group but concept cells differ; with the code's groups and the height rule
the low ACG group does not differ. The outcome rests on one of the readings the criterion names.

## fig4d-maintenance-bursts-vs-poisson

> During maintenance, bursting exceeded the Poisson null for all neurons pooled and for the low and
> high mean ACG and short and long tau decay groups, but not for concept cells.

Maintenance windows of 2.8 s from onset (the code's), each trial cut at the probe.

In [ ]:
BURST_M = show_bursts("maint")
print(BURST_M.round(4).to_string(index=False))
for reading, o in BURST_M.groupby("reading", sort=False):
    ok = {g: bool((o[o.group == g].p_BH < 0.05).iloc[0] and (o[o.group == g].real_above > o[o.group == g].n / 2).iloc[0])
          for g in ["All902", "LowACG", "HighACG", "ShortDecay", "LongDecay"]}
    print(f"{reading}: five named groups above the null {all(ok.values())} {ok}; concept cells corrected p "
          f"{o[o.group == 'Concept'].p_BH.iloc[0]:.3f}")

In [ ]:
# Variant: each trial's own maintenance length concatenated (stride = its duration), main reading.
rows = []
for sub_g, subs in GR["code"].items():
    for sub, nids in subs.items():
        t = T[sub][T[sub].load == 1]
        durs = t.maint_dur.values
        trains = [wm.epoch_spikes(SPK[n], t.maint.values, t.probe.values, 10.0) for n in nids]
        offs = np.r_[0, np.cumsum(durs)[:-1]]
        pooled = np.concatenate([np.concatenate([s + o for s, o in zip(tr_, offs)]) for tr_ in trains])
        sm = wm.smooth_counts(pooled, durs.sum())
        thr = np.percentile(sm, wm.BURST_PCT)
        rc = len(wm.burst_peaks(sm, thr, "prominence", 2))
        rng = np.random.default_rng(wm.stable_seed("own", sub_g, sub))
        counts = np.array([[len(s) for s in tr_] for tr_ in trains])
        sur = []
        for _ in range(100):
            nn = rng.poisson(counts)
            ti = np.repeat(np.tile(np.arange(len(durs)), nn.shape[0]), nn.ravel())
            st_ = rng.uniform(0, 1, len(ti)) * durs[ti] + offs[ti]
            s2 = wm.smooth_counts(st_, durs.sum())
            sur.append(len(wm.burst_peaks(s2, thr, "prominence", 2)))
        rows.append(dict(group=sub_g, sub=sub, real=rc, sur=np.mean(sur)))
print("each trial's own maintenance, code groups, prominence, real threshold:")
print(real_vs_null(pd.DataFrame(rows)).round(4).to_string(index=False))

**Result: reproduced.** Under the real-threshold readings (prominence or height, both group
rules) the five named groups have more real than surrogate bursts in most participants with BH
p at most 0.047, and concept cells do not differ (BH p 0.11 to 0.89). With each surrogate's own
threshold the pattern fails. The variant with each trial's own maintenance length keeps four of
the five (short tau decay BH p 0.11).

## fig4-acg-group-differences

> The low mean ACG group burst more than the high mean ACG group in both encoding and maintenance,
> whereas the short tau decay group exceeded the long tau decay group only during encoding.

Real burst counts of the two previous sections compared between groups within participant
(paired two-sided Wilcoxon), BH over the code's 28 pairs of 8 groups, and over the four tests
named here.

In [ ]:
for (grule, rule, dist, thr), df in BR.groupby(["grule", "rule", "dist", "thr"]):
    if dist != 2 or thr != "real":
        continue
    res = {}
    for epoch in ["enc", "maint"]:
        x = df[df.epoch == epoch].pivot(index="sub", columns="group", values="real")
        pairs = list(itertools.combinations(FAM, 2))
        ps = {}
        for a, b in pairs:
            xx = x[[a, b]].dropna()
            try:
                ps[(a, b)] = wilcoxon(xx[b], xx[a], zero_method="wilcox").pvalue if len(xx) >= 2 else np.nan
            except ValueError:
                ps[(a, b)] = 1.0
        q28 = dict(zip(ps, wm.bh(np.nan_to_num(list(ps.values()), nan=1.0))))
        res[epoch] = (x, ps, q28)
    four = [("enc", "LowACG", "HighACG"), ("maint", "LowACG", "HighACG"), ("enc", "ShortDecay", "LongDecay"), ("maint", "ShortDecay", "LongDecay")]
    p4 = [res[e][1][(a, b)] for e, a, b in four]
    q4 = wm.bh(p4)
    print(f"reading {grule}/{rule}:")
    for (e, a, b), p, q in zip(four, p4, q4):
        x = res[e][0][[a, b]].dropna()
        print(f"  {e}: {a} median {x[a].median():.1f} vs {b} {x[b].median():.1f}, {a} higher in {(x[a] > x[b]).sum()} of {len(x)}; "
              f"p = {p:.4f}, BH over 28 pairs {res[e][2][(a, b)]:.4f}, over these four {q:.4f}")

**Result: partially reproduced.** Under the prominence rule the low ACG group bursts more than
the high ACG group in both epochs, in all 12 participants (p = 0.0005, BH p at most 0.0017), and
short and long tau decay do not differ in maintenance; but short tau decay does not exceed long
tau decay in encoding (p 0.14 with the code's groups, 0.086 with the Methods'). Under the height
rule the low ACG group bursts less than the high ACG group.

## fig4f-all-cells-epochs

> At the population level, load 1 burst rates increased from encoding to each of the three 1 s
> maintenance windows and to the probe.

Burst detection of fig4c per epoch (encoding and probe 1 s, maintenance up to 3 s); M1 to M3 are
(a) the Methods' 1 s windows from maintenance onset (peaks counted by their time in the trial; M3
runs from 2 s to the end of maintenance, 0.5 to 0.8 s) or (b) the code's thirds of the whole
concatenated train. Paired two-sided Wilcoxon, BH over the 10 pairs of periods, per group and load.

In [ ]:
PER = ["enc", "M1", "M2", "M3", "probe"]
rows = []
for grule, G in GR.items():
    for rule in ["prominence", "height"]:
        for mr in ["methods", "code"]:
            for L in [1, 2, 3]:
                for g, subs in G.items():
                    for sub, nids in subs.items():
                        c = wm.epoch_burst_counts({"enc": [EP[(n, L, "enc")] for n in nids],
                                                   "maint": [EP[(n, L, "maint3")] for n in nids],
                                                   "probe": [EP[(n, L, "probe")] for n in nids]}, rule, 2, mr)
                        rows.append(dict(grule=grule, rule=rule, mread=mr, load=L, group=g, sub=sub, **c))
EPB = pd.DataFrame(rows)


def epoch_tests(x):
    ps, names = [], []
    for a, b in itertools.combinations(PER, 2):
        try:
            p = wilcoxon(x[a], x[b], zero_method="wilcox").pvalue
        except ValueError:
            p = 1.0
        ps.append(p)
        names.append(f"{a}-{b}")
    q = wm.bh(ps)
    return dict(zip(names, q)), dict(zip(names, ps))


def epoch_table(group, load):
    out = []
    for (grule, rule, mr), x in EPB[(EPB.group == group) & (EPB.load == load)].groupby(["grule", "rule", "mread"]):
        q, _ = epoch_tests(x)
        med = x[PER].median()
        out.append({"reading": f"{grule}/{rule}/M-{mr}", "n": len(x), **{f"med_{k}": med[k] for k in PER},
                    **{f"q_{k}": round(q[f"enc-{k}"], 4) for k in PER[1:]},
                    "min_q_other": round(min(v for k, v in q.items() if not k.startswith("enc-")), 4),
                    "q_M1-M2": round(q["M1-M2"], 4), "q_M1-M3": round(q["M1-M3"], 4)})
    return pd.DataFrame(out)


for g in ["All902", "AllFit"]:
    print(f"\n{g}, load 1 (medians are bursts per window; q is BH over the 10 pairs):")
    print(epoch_table(g, 1).to_string(index=False))
x = EPB[(EPB.group == "All902") & (EPB.load == 1) & (EPB.mread == "methods")]
dur = np.mean([T[s][T[s].load == 1].maint_dur.mean() for s in x["sub"].unique()]) - 2.0
print(f"\nM3 under reading (a) lasts {dur:.2f} s on average; its median rate per second, all neurons, prominence: "
      f"{(x[(x.grule == 'code') & (x.rule == 'prominence')].M3 / dur).median():.1f} against M1 {x[(x.grule == 'code') & (x.rule == 'prominence')].M1.median():.1f}")

**Result: partially reproduced.** Under the prominence rule all 902 neurons burst more in each of
M1, M2 and M3 than in encoding (medians 6 against 14 to 17, BH p 0.0003 to 0.035) under both
window readings, but the probe does not differ from encoding under any reading (BH p 0.61 to
0.71; medians 6.0 and 5.5). Three of the four comparisons hold. Between maintenance windows, M1
and M3 differ only under the Methods' windows, where M3 is shorter (0.67 s).

## fig4hj-group-epochs

> The increase of bursting from encoding into maintenance was driven by neurons with high mean ACG
> amplitude (also increasing to the probe) and neurons with a long tau decay, while the low ACG,
> short tau decay and concept-cell groups showed no change across epochs.

In [ ]:
for g in ["HighACG", "LongDecay", "LowACG", "ShortDecay", "Concept"]:
    print(f"\n{g}, load 1:")
    print(epoch_table(g, 1).to_string(index=False))

In [ ]:
for (grule, rule, mr), _ in EPB.groupby(["grule", "rule", "mread"]):
    q = {g: epoch_tests(EPB[(EPB.group == g) & (EPB.load == 1) & (EPB.grule == grule) & (EPB.rule == rule) & (EPB.mread == mr)])[0]
         for g in ["HighACG", "LongDecay", "LowACG", "ShortDecay", "Concept"]}
    med = {g: EPB[(EPB.group == g) & (EPB.load == 1) & (EPB.grule == grule) & (EPB.rule == rule) & (EPB.mread == mr)][PER].median()
           for g in ["HighACG", "LongDecay"]}
    hi = {k: bool(q["HighACG"][f"enc-{k}"] < 0.05 and med["HighACG"][k] > med["HighACG"]["enc"]) for k in ["M1", "M2", "probe"]}
    lo = {k: bool(q["LongDecay"][f"enc-{k}"] < 0.05 and med["LongDecay"][k] > med["LongDecay"]["enc"]) for k in ["M1", "M2", "M3"]}
    none = {g: int(sum(v < 0.05 for v in q[g].values())) for g in ["LowACG", "ShortDecay", "Concept"]}
    print(f"{grule}/{rule}/M-{mr}: high ACG up at {hi}; long tau decay up at {lo}; significant pairs in low ACG, short tau decay, concept {none}")

**Result: partially reproduced.** With the prominence rule and the code's thirds (either group
rule) long tau decay neurons increase from encoding to M1, M2 and M3 and the low ACG, short tau
decay and concept groups have no significant pair, but high ACG neurons increase at M1 and M2 and
not at the probe (BH p 0.24 to 0.38). The increases hold for one of the two groups.

## ext2-bursts-loads23

> The patterns of burst rates across epochs held for loads 2 and 3.

"Similar" read as all cells and the high ACG and long tau decay groups showing more bursts in
each maintenance window than in encoding, in the median over participants, at each load
(encoding from the last encoding image).

In [ ]:
rows = []
for (grule, rule, mr), x0 in EPB.groupby(["grule", "rule", "mread"]):
    for L in [1, 2, 3]:
        ok = {}
        for g in ["All902", "HighACG", "LongDecay"]:
            x = x0[(x0.group == g) & (x0.load == L)]
            med = x[PER].median()
            ok[g] = bool(all(med[k] > med["enc"] for k in ["M1", "M2", "M3"]))
            q, _ = epoch_tests(x)
            ok[g + "_q"] = "/".join(f"{q[f'enc-{k}']:.3f}" for k in ["M1", "M2", "M3", "probe"])
        rows.append({"reading": f"{grule}/{rule}/M-{mr}", "load": L, **ok})
e2 = pd.DataFrame(rows)
print(e2.to_string(index=False))

**Result: reproduced.** Under the Methods' groups with the prominence rule and the code's
thirds, all cells, high ACG and long tau decay neurons have more bursts in each maintenance
window than in encoding at loads 2 and 3, as at load 1. Under the code's groups the long tau
decay group does not at loads 2 and 3; under the height rule the pattern does not hold.

## fig5kl-concept-cell-types

> Of the classified concept cells, 11 were putative pyramidal and 14 putative interneurons;
> pyramidal concept cells showed pronounced bursting, interneuron concept cells smoother regular
> spiking.

In [ ]:
for reading in ["a", "b"]:
    ty = TYPES[reading]
    F = FEAT[reading]
    py, inn = CONCEPT & (ty == "PY"), CONCEPT & (ty == "IN")
    print(f"reading ({reading}): concept cells PY {py.sum()}, IN {inn.sum()}, unclassified {CONCEPT.sum() - py.sum() - inn.sum()}; "
          f"Royer burst index median PY {F.burst_royer[py].median():.3f}, IN {F.burst_royer[inn].median():.3f}; "
          f"ISI CV median PY {F.cv_isi[py].median():.3f}, IN {F.cv_isi[inn].median():.3f}; "
          f"Holt CV2 median PY {F.cv2_holt[py].median():.3f}, IN {F.cv2_holt[inn].median():.3f}")
print("paper: PY 11 (9.9 to 12.1), IN 14 (12.6 to 15.4)")

**Result: reproduced.** Under reading (b) of the types, 11 concept cells are PY and 14 IN, the
paper's counts, and PY concept cells have the higher Royer burst index (median 0.949 against
0.390) and Holt CV2 (1.217 against 0.991). Under reading (a) the counts are 11 and 16 (IN outside
tolerance), with the same ordering.

## fig6ab-celltype-bursts-vs-null

> Pyramidal populations showed elevated bursting over a null in both encoding and maintenance,
> whereas interneurons showed elevated bursting during maintenance but not during encoding.

Null readings: (a) the Poisson surrogates of fig4c (the PY and IN rows of its tables, BH over
the code's eight groups); (b) shuffling the cell-type labels among each participant's classified
neurons, group sizes kept (100 shuffles), BH over the two groups.

In [ ]:
for epoch, tab in [("enc", BURST_ENC), ("maint", BURST_M)]:
    x = tab[tab.group.isin(["PY", "IN"])]
    print(f"(a) Poisson null, {epoch}:")
    print(x[["reading", "group", "n", "real_median", "null_median", "real_above", "W", "p", "p_BH"]].round(4).to_string(index=False))
rows = []
for epoch, key, win in [("enc", "enc", 1.0), ("maint", "maint28", 2.8)]:
    subs = sorted(set(GR["code"]["PY"]) | set(GR["code"]["IN"]))
    for sub in subs:
        nids = NT.nid[NT["sub"] == sub].values
        pyn = [n for n in nids if TY[n] == "PY"]
        inn = [n for n in nids if TY[n] == "IN"]
        if len(pyn) == 0 or len(inn) == 0:
            continue
        rng = np.random.default_rng(wm.stable_seed("typeshuffle", epoch, sub))
        npy, nin = wm.type_shuffle_null([EP[(n, 1, key)] for n in pyn], [EP[(n, 1, key)] for n in inn], win, rng)
        for g, nl, members in [("PY", npy, pyn), ("IN", nin, inn)]:
            if len(members) >= 3:
                rc, _, _, _ = wm.group_bursts([EP[(n, 1, key)] for n in members], win, rng, "prominence", 2, "real", n_sur=0)
                rows.append({"epoch": epoch, "group": g, "sub": sub, "real": rc, "sur": nl})
SH = pd.DataFrame(rows)
for epoch in ["enc", "maint"]:
    o = real_vs_null(SH[SH.epoch == epoch], groups=["PY", "IN"], extra=())
    print(f"(b) label-shuffled null, {epoch}:")
    print(o.round(4).to_string(index=False))

**Result: partially reproduced.** Against the Poisson null with the prominence rule, PY exceed the
null in encoding (BH p 0.0089) and maintenance (0.0035) and IN in maintenance (0.0012), but IN
also exceed it in encoding (0.020), where the paper finds no difference. With the height rule PY
do not differ in encoding. Against the label-shuffled null, PY exceed it in both epochs (BH p
0.0029 and 0.0057), IN fall below it in encoding (10 against 11.9, BH p 0.0016) and do not differ
in maintenance (0.063). No reading gives the paper's full pattern; the elevated PY bursting and IN
maintenance bursting hold.

## fig6cd-celltype-epochs

> Pyramidal burst rates stayed stable across task epochs at every load, whereas interneuron burst
> rates changed from encoding to each maintenance window and to the probe in load 1 trials.

In [ ]:
for g in ["PY", "IN"]:
    for L in [1, 2, 3]:
        print(f"\n{g}, load {L}:")
        print(epoch_table(g, L).to_string(index=False))

In [ ]:
for (grule, rule, mr), x0 in EPB.groupby(["grule", "rule", "mread"]):
    if grule != "code":
        continue
    qin = epoch_tests(x0[(x0.group == "IN") & (x0.load == 1)])[0]
    py_sig = {L: int(sum(v < 0.05 for v in epoch_tests(x0[(x0.group == "PY") & (x0.load == L)])[0].values())) for L in [1, 2, 3]}
    print(f"{rule}/M-{mr}: IN encoding vs M1, M2, M3, probe corrected p "
          f"{[round(qin[f'enc-{k}'], 4) for k in ['M1', 'M2', 'M3', 'probe']]}; PY pairs with corrected p < 0.05 by load {py_sig}")

**Result: partially reproduced.** With the prominence rule and the code's thirds, PY have no
pair of periods with BH p < 0.05 at any load, and IN burst more in M1, M2 and M3 than in encoding
at load 1 (BH p 0.0047 to 0.014), but IN do not differ between encoding and the probe (0.80). The
Methods' windows make PY differ. The IN change is an increase (medians 11 to 15).

## ext5bd-in-load-dependence

> Interneuron burst-rate differences across maintenance epochs increased with load.

Read as the median over participants of the absolute difference in bursts between encoding and
each of M1 to M3, averaged over the three windows.

In [ ]:
for (grule, rule, mr), x0 in EPB.groupby(["grule", "rule", "mread"]):
    vals = []
    for L in [1, 2, 3]:
        x = x0[(x0.group == "IN") & (x0.load == L)]
        vals.append(np.median(np.mean([np.abs(x[k] - x["enc"]) for k in ["M1", "M2", "M3"]], 0)))
    print(f"{grule}/{rule}/M-{mr}: IN measure by load {np.round(vals, 2).tolist()}; rises from load 1 to 3: {vals[2] > vals[0]}")

**Result: reproduced.** With the prominence rule and the code's thirds (either group rule) the
measure rises from 3.67 at load 1 to 5.00 and 6.33; under the other readings it does not rise.

## fig6e-load-modulation-in

> A significant portion of interneurons was modulated by memory load during maintenance (14.36%),
> including interneuron concept cells (21.42%), while pyramidal neurons showed minimal load
> modulation (5.43%; concept pyramidal 0.00%).

The GLM fits of fig1e (load, every trial reading), with the groups PY, IN, PY concept cells and IN
concept cells (types of reading (a) of fig5c).

In [ ]:
TG = {"IN": np.where(TY == "IN")[0], "IN concept": np.where((TY == "IN") & CONCEPT)[0],
      "PY": np.where(TY == "PY")[0], "PY concept": np.where((TY == "PY") & CONCEPT)[0]}
for key in [("load", "matched", "shared"), ("load", "matched", "within"), ("load", "all", "within")]:
    (real, null), _, _ = GLM[key]
    df, _ = wm.glm_groups(real, null, TG)
    print(f"load, trials {key[1]}, permutation {key[2]}:")
    print(df.round(3).to_string(index=False))
TGb = {"IN": np.where(TYPES["b"] == "IN")[0], "IN concept": np.where((TYPES["b"] == "IN") & CONCEPT)[0],
       "PY": np.where(TYPES["b"] == "PY")[0], "PY concept": np.where((TYPES["b"] == "PY") & CONCEPT)[0]}
(real, null), _, _ = GLM[("load", "matched", "shared")]
print("variant, types of reading (b), matched trials, shared permutation:")
print(wm.glm_groups(real, null, TGb)[0].round(3).to_string(index=False))
print("paper: IN 14.36% (P = 0.002), IN concept 21.42% (P = 0.032), PY 5.43% and PY concept 0.00% (P > 0.05)")

**Result: partially reproduced.** IN are load-modulated (13.95%, p = 0.001, matched trials and
shared permutation, within tolerance) and PY and PY concept cells are not (p 0.43 and 1.0), but IN
concept cells give 18.75% (3 of 16, p = 0.042), below the tolerance of 19.3 to 23.6%; the paper's
21.42% is 3 of 14. With reading (b) of the types (14 IN concept cells) 2 are significant (p = 0.15).

## fig6f-rt-modulation-in-concept

> Interneuron concept cells were modulated by reaction time during maintenance (35.71%), but not
> all interneurons, and pyramidal neurons showed no significant RT modulation in either the full
> population or the concept subset.

In [ ]:
for key in [("rt", "matched", "shared"), ("rt", "matched", "within"), ("rt", "all", "within"),
            ("rt_code", "matched", "shared"), ("rt_code", "matched", "within")]:
    (real, null), _, _ = GLM[key]
    df, _ = wm.glm_groups(real, null, TG)
    label = "own RT" if key[0] == "rt" else "variant, subject 1's RT vector"
    print(f"{label}, trials {key[1]}, permutation {key[2]}:")
    print(df.round(3).to_string(index=False))
print("paper: IN concept 35.71% (P = 0.002); IN, PY, PY concept P > 0.05")

**Result: not reproduced.** With each trial's own RT, IN concept cells give 12.50% (p = 0.19)
and PY are significant (p 0.005 to 0.024). The code's predictor (subject 1's RT vector) gives IN
concept cells 31.25% (p = 0.001), just below the tolerance of 32.1 to 39.3%, with all IN (p 0.086)
and PY not significant under the shared permutation.

## fig6gh-in-concept-cv2

> Interneuron concept cells had higher CV2 during maintenance than encoding on preferred load 1
> trials but not on nonpreferred trials, while pyramidal concept cells showed stable CV2.

Holt's CV2 of the interspike intervals of each cell's spikes concatenated over preferred (or
nonpreferred) trials, in encoding (first image, 1 s) and maintenance (2.5 s); trials of load 1
(Results) or of all loads (the code). Readings for PY: (a) encoding against maintenance within
each trial type; (b) preferred against nonpreferred within each epoch.

In [ ]:
def cv2_cell(n, loads, pref_flag, epoch):
    t = T[NT["sub"].iat[n]]
    t = t[t.load.isin(loads)]
    pm = pref_mask(t, PREF[n], 1) if loads == [1] else ((t.img1 == PREF[n]) | ((t.load >= 2) & (t.img2 == PREF[n])) | ((t.load == 3) & (t.img3 == PREF[n]))).values
    t = t[pm if pref_flag else ~pm]
    on, win = (t.enc1.values, 1.0) if epoch == "enc" else (t.maint.values, 2.5)
    sp = wm.epoch_spikes(SPK[n], on, on + win, win)
    return wm.cv2_holt(np.diff(wm.concat_trials(sp, win)))


for loads, lname in [([1], "load 1"), ([1, 2, 3], "all loads")]:
    for ty_ in ["IN", "PY"]:
        cells = np.where(CONCEPT & (TY == ty_))[0]
        vals = {(pf, ep): np.array([cv2_cell(n, loads, pf, ep) for n in cells]) for pf in [True, False] for ep in ["enc", "maint"]}
        for pf in [True, False]:
            a, b = vals[(pf, "enc")], vals[(pf, "maint")]
            ok = np.isfinite(a) & np.isfinite(b)
            w = wilcoxon(b[ok], a[ok])
            print(f"{lname}, {ty_} concept, {'preferred' if pf else 'nonpreferred'}: n {ok.sum()}, CV2 median encoding {np.median(a[ok]):.3f}, "
                  f"maintenance {np.median(b[ok]):.3f}, maintenance higher in {(b[ok] > a[ok]).sum()}; W = {w.statistic:.1f}, p = {w.pvalue:.4f}")
        if ty_ == "PY":
            for ep in ["enc", "maint"]:
                a, b = vals[(True, ep)], vals[(False, ep)]
                ok = np.isfinite(a) & np.isfinite(b)
                w = wilcoxon(a[ok], b[ok])
                print(f"   reading (b), PY {ep}: preferred vs nonpreferred W = {w.statistic:.1f}, p = {w.pvalue:.4f}")
print("paper: IN preferred W = 13.0, P = 0.032; IN nonpreferred W = 50.0, P > 0.05; PY all P > 0.05")

**Result: not reproduced.** IN concept cells do not have higher CV2 in maintenance than in
encoding on preferred trials (load 1 p = 0.86; all loads p = 0.13). The PY and nonpreferred parts
hold (no comparison with p < 0.05).

## ext5gh-py-decay-preferred

> Pyramidal concept cells showed increased decay time constants on preferred trials during encoding
> and maintenance, an effect absent in interneurons.

tau_decay of the CellExplorer fit (acg-fit-yield) to the narrow ACG of each concept cell's spikes
in the epoch of preferred (or nonpreferred) trials, concatenated; preferred against nonpreferred
by a two-sided Wilcoxon signed-rank test over cells (a paired t-test as a variant); load 1 or all
loads. The claims file takes this reading of "decay time constants" and the outcome rests on it.

In [ ]:
def decay_cell(n, loads, pref_flag, epoch):
    t = T[NT["sub"].iat[n]]
    t = t[t.load.isin(loads)]
    pm = ((t.img1 == PREF[n]) | ((t.load >= 2) & (t.img2 == PREF[n])) | ((t.load == 3) & (t.img3 == PREF[n]))).values
    if loads == [1]:
        pm = (t.img1 == PREF[n]).values
    t = t[pm if pref_flag else ~pm]
    on, win = (t.enc1.values, 1.0) if epoch == "enc" else (t.maint.values, 2.5)
    s = wm.concat_trials(wm.epoch_spikes(SPK[n], on, on + win, win), win)
    if len(s) < 3:
        return np.nan, len(s)
    f = wm.fit_acg(wm.acg(np.sort(s), 0.0005, 201))
    return f["tau_decay"], len(s)


for loads, lname in [([1], "load 1"), ([1, 2, 3], "all loads")]:
    for ty_ in ["PY", "IN"]:
        cells = np.where(CONCEPT & (TY == ty_))[0]
        for ep in ["enc", "maint"]:
            a = np.array([decay_cell(n, loads, True, ep) for n in cells])
            b = np.array([decay_cell(n, loads, False, ep) for n in cells])
            ok = np.isfinite(a[:, 0]) & np.isfinite(b[:, 0])
            w = wilcoxon(a[ok, 0], b[ok, 0])
            tt = stats.ttest_rel(a[ok, 0], b[ok, 0])
            print(f"{lname}, {ty_} concept, {ep}: n {ok.sum()} (spikes per cell, preferred median {np.median(a[:, 1]):.0f}); "
                  f"tau_decay median preferred {np.median(a[ok, 0]):.2f} ms, nonpreferred {np.median(b[ok, 0]):.2f}; "
                  f"preferred larger in {(a[ok, 0] > b[ok, 0]).sum()}; Wilcoxon W = {w.statistic:.1f}, p = {w.pvalue:.4f}; paired t p = {tt.pvalue:.4f}")
print("paper: PY preferred larger in encoding and maintenance, Pmax = 0.029; IN all P > 0.05")

**Result: not reproduced.** Under the reading of the claims file (ACG tau_decay on preferred and
nonpreferred spikes), PY concept cells show no preferred-trial increase in either epoch (Wilcoxon
p 0.13 to 0.73); IN show none either (p 0.058 to 0.67). The outcome rests on that reading.

## fig6i-py-bin-width-decoding

> Pyramidal concept cells did not decode the image above chance during load 1 maintenance with 250
> and 500 ms bins (26.67%, 23.33%) but did with 750 and 1000 ms bins (40.00%, 43.33%).

The analysis of fig2f with the PY concept cells as the pseudo-population, BH over 12 tests.

In [ ]:
CPY = np.where(CONCEPT & (TY == "PY"))[0]
CIN = np.where(CONCEPT & (TY == "IN"))[0]
BW["PY"] = bw_run("PY", CPY)
BW["IN"] = bw_run("IN", CIN)
print(f"PY concept cells {len(CPY)}, IN concept cells {len(CIN)}")
print(BW["PY"][1].to_string(index=False))
print("paper: 26.67, 23.33 (P > 0.05), 40.00, 43.33% (P = 0.030)")

**Result: not reproduced.** With the code's table PY concept cells give 33.33, 23.33, 36.67 and
43.33%, but none survives BH (p 0.060 to 0.39), so the 750 and 1000 ms accuracies are not
significant; with reading (b) the pattern is reversed (250 and 500 ms highest).

## fig6j-in-bin-width-decoding

> Interneuron concept cells decoded the image above chance during load 1 maintenance at all bin
> widths (53.33%, 50.00%, 46.67%, 53.33%), less sensitive to bin width than pyramidal cells.

In [ ]:
print(BW["IN"][1].to_string(index=False))
print("paper: 53.33, 50.00, 46.67, 53.33% (P = 0.002)")

**Result: partially reproduced.** IN concept cells decode above chance at every width under both
trial readings (BH p 0.002), less dependent on width than PY, but two of the four accuracies are
outside tolerance under each reading (code's table 46.67, 53.33, 53.33 and 50.00%).

## fig6kl-cross-temporal-by-type

> Cross-temporal decoding showed no above-chance generalization for the 11 pyramidal concept
> cells, while interneuron concept cells decoded above chance in cross-temporal maps.

In [ ]:
CT[("PY", 1, "first")] = ct_analysis(CPY, 1, "first")
CT[("IN", 1, "first")] = ct_analysis(CIN, 1, "first")
print(pd.concat([summarize_ct(CT[("PY", 1, "first")], "PY concept, load 1"),
                 summarize_ct(CT[("IN", 1, "first")], "IN concept, load 1")]).to_string(index=False))

**Result: reproduced.** The PY concept map has no significant pair under any reading, and the IN
concept map keeps 162 to 230 off-diagonal pairs.

## ext5ij-load2-decoding-by-type

> Interneuron concept cells also decoded the second image in load 2 trials above chance at all bin
> widths, while pyramidal concept cells did not.

In [ ]:
for g in ["IN", "PY"]:
    df = BW[g][1]
    print(f"{g} concept cells, load 2:")
    print(df[df.load == 2].to_string(index=False))

**Result: reproduced.** At load 2 IN concept cells decode above chance at all four widths under
both readings (BH p at most 0.013) and PY concept cells at none (BH p at least 0.49).

In [ ]:
print(f"notebook ran in {(time.time() - T0) / 60:.1f} min")

## Summary

| Claim | Outcome |
|---|---|
| units-by-region | reproduced |
| behavior-accuracy | reproduced |
| fig1c-accuracy-load1-vs-load3 | reproduced |
| behavior-rt | reproduced |
| rt-in-vs-out-memory | reproduced |
| rt-correct-vs-incorrect | reproduced |
| fig1e-load-modulated-neurons | reproduced |
| fig1f-rt-modulation-not-significant | not reproduced |
| concept-cells-count | reproduced |
| fig2ab-single-cells-not-persistent | reproduced |
| fig2c-population-persistence | reproduced |
| ext1bc-persistence-by-load | reproduced |
| fig2d-cross-temporal-localized | reproduced |
| ext1de-cross-temporal-load23 | reproduced |
| fig2e-maintenance-decoding-amy-mtl | reproduced |
| ext1f-maintenance-decoding-load2 | partially reproduced |
| fig2f-bin-width-decoding | partially reproduced |
| fig2f-bin-width-pairwise | not reproduced |
| ext1g-bin-width-load2 | reproduced |
| fig3abc-trajectory-complexity | blocked (too large) |
| fig3d-encoding-vs-maintenance-separability | blocked (too large) |
| fig3h-early-vs-late-maintenance | blocked (too large) |
| fig4c-encoding-bursts-vs-poisson | reproduced |
| fig4d-maintenance-bursts-vs-poisson | reproduced |
| fig4-acg-group-differences | partially reproduced |
| fig4f-all-cells-epochs | partially reproduced |
| fig4hj-group-epochs | partially reproduced |
| ext2-bursts-loads23 | reproduced |
| acg-fit-yield | reproduced |
| fig5c-cell-type-clusters | reproduced |
| fig5dfh-cluster-features | reproduced |
| fig5gh-burst-index-cv2 | reproduced |
| supp4-rodent-validation | reproduced (on the features the code ships) |
| fig5kl-concept-cell-types | reproduced |
| fig6ab-celltype-bursts-vs-null | partially reproduced |
| fig6cd-celltype-epochs | partially reproduced |
| ext5bd-in-load-dependence | reproduced |
| fig6e-load-modulation-in | partially reproduced |
| fig6f-rt-modulation-in-concept | not reproduced |
| fig6gh-in-concept-cv2 | not reproduced |
| ext5gh-py-decay-preferred | not reproduced |
| fig6i-py-bin-width-decoding | not reproduced |
| fig6j-in-bin-width-decoding | partially reproduced |
| fig6kl-cross-temporal-by-type | reproduced |
| ext5ij-load2-decoding-by-type | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The ACG fit is a Python port of the code's
MATLAB functions, with scipy's optimizer, so R2 and the time constants differ for part of the
neurons and the cell-type counts differ slightly (102 or 100 PY against 92); later analyses use
these types. The concept-cell bootstrap is run with fixed seeds (the code sets none); the set used
shares 57 of 58 cells with the code's list. The SVM is libsvm through sklearn's wrapper on a
precomputed linear kernel, checked equal to `SVC(kernel='linear')`. The Poisson GLM is fitted by
Newton's method, checked against sklearn. Many method details the paper leaves open were run
under each reading the claims file names; outcomes that rest on one of several readings say so.
The dPCA analyses were not run (too large).

What was found about the dataset. The dandiset holds all 902 units of the paper with the
paper's region counts, and the paper's code tables link to its trials exactly. The paper's
description differs from the files in the number of sessions with 108 trials (six, not five), the
fixation durations (up to 1.2 s and longer, not 0.9 to 1.0 s), and the image duration, which
varies by session (1.0 to 2.5 s) and is not stated. One trial (subject 9) has a maintenance of 0 s.
The paper's RT-modulation percentages (Fig. 1f) are reproduced exactly only with the code's
predictor, which pairs every neuron with subject 1's reaction times.